In [ ]:
"""
============================================================
  TDNN (Time Delay Neural Network) — Javanese ASR
  Perbandingan dengan GMM pipeline
============================================================
  Label    : Word-level per frame (uniform segmentation)
             Vocab = kata unik dari seluruh dataset
  Input    : .wav dari SAMPLED_ROOT + dataSampledSmall.csv
  Output   : WER + Akurasi (identik format GMM)
============================================================
"""

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  KONFIGURASI
# ─────────────────────────────────────────────────────────────

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/TDNN/out/test_5"
)
os.makedirs(DIR_SAVE, exist_ok=True)

OUT_WER_CSV     = os.path.join(DIR_SAVE, "HasilWER_TDNN.csv")
OUT_DEC_CSV     = os.path.join(DIR_SAVE, "HasilDekode_TDNN.csv")
OUT_ACC_CSV     = os.path.join(DIR_SAVE, "HasilAkurasi_TDNN.csv")
OUT_HISTORY_CSV = os.path.join(DIR_SAVE, "HasilTraining_TDNN.csv")
MODEL_PATH      = os.path.join(DIR_SAVE, "tdnn_best.npy")

COL_PATH_DATA = "Path_Data"
COL_SENTENCES = "sentences"

# ── MFCC (identik GMM) ──
SR=16000; N_MFCC=13; N_FFT=512; HOP_LENGTH=160
WIN_LENGTH=400; N_MELS=26; PRE_EMPH=0.97
FMIN=0.0; FMAX=4000.0; DELTA_WIDTH=2
INPUT_DIM = N_MFCC * 3   # 39

# ── Arsitektur ──
TDNN_CONTEXTS = [[-2,2],[-2,0],[-3,3],[0,0]]
HIDDEN_DIMS   = [512, 512, 512, 512]
DROPOUT_RATE  = 0.1
LEARNING_RATE = 0.001

# ── Training ──
BATCH_SIZE  = 512    # frame per batch
MAX_EPOCHS  = 30
EARLY_STOP  = 5
GRAD_CLIP   = 5.0

# ── Split ──
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
RANDOM_SEED = 42


# ─────────────────────────────────────────────────────────────
#  EKSTRAKSI MFCC — VECTORIZED (identik GMM, tanpa Python loop)
# ─────────────────────────────────────────────────────────────

try:
    import librosa
    LIBROSA_OK = True
except ImportError:
    LIBROSA_OK = False
try:
    from scipy.io import wavfile
    SCIPY_OK = True
except ImportError:
    SCIPY_OK = False


def extract_features(wav_path: str) -> np.ndarray:
    """Ekstrak MFCC 39-dim (13 + delta + delta2) dari .wav."""
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(wav_path)
    return _librosa(wav_path) if LIBROSA_OK else _manual(wav_path)


def _librosa(wav_path):
    y, _  = librosa.load(wav_path, sr=SR, mono=True)
    if len(y) == 0:
        raise ValueError("Audio kosong")
    y     = np.append(y[0], y[1:] - PRE_EMPH * y[:-1])
    mfcc  = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT,
        hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, window="hamming"
    )  # (13, T)
    d1 = librosa.feature.delta(mfcc, width=DELTA_WIDTH*2+1)
    d2 = librosa.feature.delta(mfcc, order=2, width=DELTA_WIDTH*2+1)
    return np.vstack([mfcc, d1, d2]).T.astype(np.float32)  # (T,39)


def _delta_vec(feat: np.ndarray, width: int = 2) -> np.ndarray:
    """Delta vectorized — tanpa Python loop antar frame."""
    T     = feat.shape[0]
    denom = 2.0 * sum(n*n for n in range(1, width+1))
    delta = np.zeros_like(feat)
    for n in range(1, width+1):
        # geser maju dan mundur, clamp di tepi
        fwd = feat[np.clip(np.arange(T)+n, 0, T-1)]
        bwd = feat[np.clip(np.arange(T)-n, 0, T-1)]
        delta += n * (fwd - bwd)
    return delta / denom


def _manual(wav_path):
    if not SCIPY_OK:
        raise ImportError("Butuh librosa atau scipy")
    sr, audio = wavfile.read(wav_path)
    if audio.dtype == np.int16:
        audio = audio / 32768.0
    elif audio.dtype == np.int32:
        audio = audio / 2147483648.0
    audio = audio.astype(np.float32)
    if sr != SR:
        n = int(len(audio)*SR/sr)
        audio = np.interp(np.linspace(0,len(audio)-1,n), np.arange(len(audio)), audio)
    y     = np.append(audio[0], audio[1:] - PRE_EMPH*audio[:-1])
    # Framing vectorized
    T_fr  = 1 + (len(y) - WIN_LENGTH) // HOP_LENGTH
    idx   = (np.arange(WIN_LENGTH)[None,:] +
             np.arange(T_fr)[:,None]*HOP_LENGTH)
    frames = y[idx] * np.hamming(WIN_LENGTH)
    fft_m  = np.abs(np.fft.rfft(frames, n=N_FFT))
    # Mel filterbank
    def hz2mel(f): return 1127*np.log1p(f/700)
    def mel2hz(m): return 700*(np.exp(m/1127)-1)
    pts  = mel2hz(np.linspace(hz2mel(FMIN), hz2mel(FMAX), N_MELS+2))
    bins = np.floor((N_FFT+1)*pts/SR).astype(int)
    fb   = np.zeros((N_MELS, N_FFT//2+1))
    for m in range(1, N_MELS+1):
        for k in range(bins[m-1], bins[m]):
            fb[m-1,k] = (k-bins[m-1])/(bins[m]-bins[m-1]+1e-10)
        for k in range(bins[m], bins[m+1]):
            fb[m-1,k] = (bins[m+1]-k)/(bins[m+1]-bins[m]+1e-10)
    log_mel = np.log10(np.maximum(fft_m**2 @ fb.T, 1e-10))
    # DCT
    N_     = N_MELS
    ni     = np.arange(N_MFCC)[:,None]
    ki     = np.arange(N_)[None,:] + 0.5
    dct_m  = np.cos(np.pi * ni * ki / N_) * np.sqrt(2/N_)
    mfcc   = log_mel @ dct_m.T
    d1     = _delta_vec(mfcc, DELTA_WIDTH)
    d2     = _delta_vec(d1,   DELTA_WIDTH)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  UNIFORM WORD SEGMENTATION
# ─────────────────────────────────────────────────────────────

def uniform_word_labels(words: list, T: int, word2id: dict) -> np.ndarray:
    """
    Bagi T frame ke setiap kata secara proporsional.

    Contoh: words=["aku","arep","pasar"], T=10
      frame 0-2 → "aku", frame 3-5 → "arep", frame 6-9 → "pasar"

    Metodologi (untuk skripsi):
      Karena tidak ada forced alignment, durasi setiap kata
      diasumsikan proporsional terhadap jumlah kata (uniform).
    """
    N      = len(words)
    labels = np.zeros(T, dtype=np.int32)
    if N == 0:
        return labels
    base = T // N
    rem  = T % N
    start = 0
    for i, w in enumerate(words):
        end = start + base + (1 if i < rem else 0)
        labels[start:end] = word2id.get(w, word2id["<unk>"])
        start = end
    return labels


# ─────────────────────────────────────────────────────────────
#  DATA LOADER
# ─────────────────────────────────────────────────────────────

def find_wav(audio_id: str) -> str:
    for root, _, files in os.walk(SAMPLED_ROOT):
        for f in files:
            if os.path.splitext(f)[0] == audio_id and f.lower().endswith(".wav"):
                return os.path.join(root, f)
    return ""


def load_data(csv_path: str) -> tuple:
    """
    Baca CSV → ekstrak MFCC per frame → label kata per frame.

    Output dim = jumlah kata unik (vocab), bukan jumlah kalimat.
    """
    print(f"  Membaca {csv_path} ...")
    df = pd.read_csv(csv_path)
    print(f"  Shape CSV : {df.shape}")
    print(f"  Kolom     : {list(df.columns)}")

    for col in [COL_PATH_DATA, COL_SENTENCES]:
        if col not in df.columns:
            raise ValueError(f"Kolom '{col}' tidak ada. Ada: {list(df.columns)}")

    # ── Bangun vocab dari KATA (bukan kalimat) ────────────────
    all_words = []
    for sent in df[COL_SENTENCES].astype(str).str.lower().str.strip():
        all_words.extend(sent.split())

    vocab   = ["<unk>"] + sorted(set(all_words))
    word2id = {w: i for i, w in enumerate(vocab)}
    id2word = {i: w for w, i in word2id.items()}
    print(f"  Vocab kata unik  : {len(vocab)-1}  (+1 <unk>  → total {len(vocab)})")

    # ── Ekstrak MFCC + label per frame ───────────────────────
    sequences, labels, word_refs = {}, {}, {}
    skipped = 0

    print(f"  Mengekstrak MFCC dari {len(df)} file .wav ...")
    for idx, row in df.iterrows():
        audio_id = os.path.splitext(os.path.basename(str(row[COL_PATH_DATA])))[0]
        kalimat  = str(row[COL_SENTENCES]).lower().strip()
        words    = kalimat.split()

        wav_path = str(row[COL_PATH_DATA])
        if not os.path.isfile(wav_path):
            wav_path = find_wav(audio_id)
        if not wav_path:
            skipped += 1; continue

        try:
            feats = extract_features(wav_path)   # (T, 39)
            T     = feats.shape[0]
            if T == 0 or not words:
                skipped += 1; continue

            sequences[audio_id] = feats
            labels[audio_id]    = uniform_word_labels(words, T, word2id)
            word_refs[audio_id] = words

        except Exception as e:
            print(f"  ⚠ {audio_id}: {e}")
            skipped += 1; continue

        if (idx+1) % 200 == 0 or (idx+1) == len(df):
            print(f"    [{idx+1}/{len(df)}] diproses, {skipped} dilewati")

    total_frames = sum(v.shape[0] for v in sequences.values())
    actual_dim   = next(iter(sequences.values())).shape[1]

    print(f"\n  Audio berhasil : {len(sequences)}")
    print(f"  Audio dilewati : {skipped}")
    print(f"  Total frame    : {total_frames:,}")
    print(f"  Input dim      : {actual_dim}")
    print(f"  Output dim     : {len(vocab)}  ← vocab kata, bukan kalimat")

    return sequences, labels, word2id, id2word, word_refs, actual_dim


# ─────────────────────────────────────────────────────────────
#  AKTIVASI & LOSS
# ─────────────────────────────────────────────────────────────

def relu(x):      return np.maximum(0.0, x)
def relu_grad(x): return (x > 0).astype(np.float32)

def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / (e.sum(axis=-1, keepdims=True) + 1e-9)

def ce_loss(p, y):
    return -np.log(p[np.arange(len(y)), y] + 1e-9).mean()

def accuracy(p, y):
    return (np.argmax(p, axis=1) == y).mean()


# ─────────────────────────────────────────────────────────────
#  TDNN LAYER
# ─────────────────────────────────────────────────────────────

class TDNNLayer:
    def __init__(self, in_dim, out_dim, context, lr):
        self.context = context
        self.ctx_len = context[1] - context[0] + 1
        self.lr      = lr
        fan_in       = in_dim * self.ctx_len
        self.W = (np.random.randn(fan_in, out_dim) *
                  np.sqrt(2.0/fan_in)).astype(np.float32)
        self.b = np.zeros(out_dim, dtype=np.float32)
        self.mW=np.zeros_like(self.W); self.vW=np.zeros_like(self.W)
        self.mb=np.zeros_like(self.b); self.vb=np.zeros_like(self.b)
        self.t=0; self._X=None

    def splice(self, X):
        """Index-clamp splicing — aman untuk T=1."""
        T   = X.shape[0]
        idx = np.arange(T)
        return np.concatenate(
            [X[np.clip(idx+off, 0, T-1)]
             for off in range(self.context[0], self.context[1]+1)],
            axis=1
        )

    def forward(self, Xsp):
        self._X = Xsp
        return Xsp @ self.W + self.b

    def backward(self, dout):
        dW = self._X.T @ dout / len(dout)
        db = dout.mean(0)
        dXsp = dout @ self.W.T                        # (B, D*ctx_len)
        # Gradient clip
        norm = np.sqrt((dW**2).sum() + (db**2).sum())
        if norm > GRAD_CLIP:
            dW *= GRAD_CLIP/norm; db *= GRAD_CLIP/norm
        self._adam(dW, db)
        # Un-splice: sum kontribusi semua offset → (B, D)
        B, D_tot = dXsp.shape
        D  = D_tot // self.ctx_len
        dX = np.zeros((B, D), dtype=np.float32)
        for k in range(self.ctx_len):
            dX += dXsp[:, k*D:(k+1)*D]
        return dX

    def _adam(self, dW, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        self.mW = b1*self.mW+(1-b1)*dW
        self.vW = b2*self.vW+(1-b2)*dW**2
        self.W -= self.lr*(self.mW/(1-b1**self.t))/(np.sqrt(self.vW/(1-b2**self.t))+eps)
        self.mb = b1*self.mb+(1-b1)*db
        self.vb = b2*self.vb+(1-b2)*db**2
        self.b -= self.lr*(self.mb/(1-b1**self.t))/(np.sqrt(self.vb/(1-b2**self.t))+eps)


# ─────────────────────────────────────────────────────────────
#  MODEL TDNN
# ─────────────────────────────────────────────────────────────

class TDNNModel:
    def __init__(self, input_dim, hidden_dims, contexts,
                 output_dim, dropout=0.1, lr=0.001):
        assert len(hidden_dims) == len(contexts)
        self.dropout  = dropout
        self.training = True
        self.layers   = []
        in_d = input_dim
        for h, ctx in zip(hidden_dims, contexts):
            self.layers.append(TDNNLayer(in_d, h, ctx, lr))
            in_d = h
        self.layers.append(TDNNLayer(in_d, output_dim, [0,0], lr))
        self._rc = []; self._dc = []   # relu cache, drop cache

    def forward(self, X):
        self._rc = []; self._dc = []
        h = X
        for layer in self.layers[:-1]:
            z = layer.forward(layer.splice(h))
            a = relu(z)
            if self.training and self.dropout > 0:
                mask = (np.random.rand(*a.shape) > self.dropout
                        ).astype(np.float32)
                a    = a * mask / (1.0 - self.dropout)
            else:
                mask = np.ones(a.shape, dtype=np.float32)
            self._rc.append(z); self._dc.append(mask)
            h = a
        out = self.layers[-1]
        return out.forward(out.splice(h))

    def loss_and_backward(self, X, y):
        logits = self.forward(X)
        probs  = softmax(logits)
        loss   = ce_loss(probs, y)
        acc    = accuracy(probs, y)
        dlog   = probs.copy()
        dlog[np.arange(len(y)), y] -= 1.0
        dh  = self.layers[-1].backward(dlog)
        sc  = 1.0/(1.0-self.dropout+1e-9) if self.dropout > 0 else 1.0
        for i in reversed(range(len(self.layers)-1)):
            dh = dh * self._dc[i] * sc
            dh = dh * relu_grad(self._rc[i])
            dh = self.layers[i].backward(dh)
        return loss, acc

    def predict_proba(self, X):
        self.training = False
        return softmax(self.forward(X))

    def save(self, path):
        np.save(path, {"w": [(l.W.copy(), l.b.copy()) for l in self.layers]},
                allow_pickle=True)

    def load(self, path):
        d = np.load(path, allow_pickle=True).item()
        for i, (W, b) in enumerate(d["w"]):
            self.layers[i].W = W; self.layers[i].b = b
        print(f"  Model dimuat: {path}")


# ─────────────────────────────────────────────────────────────
#  SPLIT
# ─────────────────────────────────────────────────────────────

def split_dataset(ids):
    rng  = np.random.default_rng(RANDOM_SEED)
    ids  = list(ids)
    perm = rng.permutation(len(ids))
    n    = len(ids)
    ntr  = int(n*TRAIN_RATIO); nva = int(n*VAL_RATIO)
    tr = [ids[i] for i in perm[:ntr]]
    va = [ids[i] for i in perm[ntr:ntr+nva]]
    te = [ids[i] for i in perm[ntr+nva:]]
    print(f"  Split → train:{len(tr)}  val:{len(va)}  test:{len(te)}")
    return tr, va, te


# ─────────────────────────────────────────────────────────────
#  TRAINING
# ─────────────────────────────────────────────────────────────

def train(model, train_ids, val_ids, sequences, labels):
    rng    = np.random.default_rng(RANDOM_SEED)
    hist   = []
    best   = float("inf")
    pat    = 0

    print(f"\n  Train: {len(train_ids)} audio | Val: {len(val_ids)} audio")
    print(f"  Epochs: {MAX_EPOCHS} | EarlyStop: {EARLY_STOP} | Batch: {BATCH_SIZE} frame")
    print(f"\n  {'Ep':>4}  {'TrLoss':>8}  {'TrAcc%':>8}  {'VaLoss':>8}  {'VaAcc%':>8}  {'t':>7}")
    print(f"  {'-'*55}")

    for epoch in range(1, MAX_EPOCHS+1):
        t0 = time.time()
        model.training = True

        # Kumpulkan semua frame train → shuffle global
        all_X = np.concatenate([sequences[a] for a in train_ids])
        all_y = np.concatenate([labels[a]    for a in train_ids])
        perm  = rng.permutation(len(all_X))
        all_X = all_X[perm]; all_y = all_y[perm]

        el = ea = nb = 0
        for s in range(0, len(all_X), BATCH_SIZE):
            Xb = all_X[s:s+BATCH_SIZE]
            yb = all_y[s:s+BATCH_SIZE]
            if len(Xb) < 2: continue
            loss, acc = model.loss_and_backward(Xb, yb)
            el += loss; ea += acc; nb += 1

        el /= max(nb,1); ea /= max(nb,1)

        # Validasi
        model.training = False
        val_X = np.concatenate([sequences[a] for a in val_ids])
        val_y = np.concatenate([labels[a]    for a in val_ids])
        vp    = model.predict_proba(val_X)
        vl    = ce_loss(vp, val_y)
        va    = accuracy(vp, val_y)

        elapsed = time.time()-t0
        hist.append({
            "epoch":epoch, "train_loss":round(el,6),
            "train_acc":round(ea*100,4),
            "val_loss":round(vl,6), "val_acc":round(va*100,4)
        })
        print(f"  {epoch:4d}  {el:8.4f}  {ea*100:8.4f}  "
              f"{vl:8.4f}  {va*100:8.4f}  {elapsed:6.1f}s")

        if vl < best - 1e-5:
            best = vl; pat = 0; model.save(MODEL_PATH)
        else:
            pat += 1
            if pat >= EARLY_STOP:
                print(f"\n  ✓ Early stop epoch {epoch} (best={best:.4f})")
                break

    return hist


# ─────────────────────────────────────────────────────────────
#  DEKODING (CTC-style collapse)
# ─────────────────────────────────────────────────────────────

def decode_to_words(probs, id2word):
    """Frame predictions → collapsed word sequence."""
    preds = np.argmax(probs, axis=1)
    words, prev = [], None
    for pid in preds:
        if pid != prev:
            w = id2word.get(int(pid), "<unk>")
            if w != "<unk>":
                words.append(w)
            prev = pid
    return words


# ─────────────────────────────────────────────────────────────
#  WER (Levenshtein)
# ─────────────────────────────────────────────────────────────

def levenshtein(ref, hyp):
    R, H = len(ref), len(hyp)
    dp   = np.zeros((R+1,H+1), dtype=int)
    dp[:,0] = np.arange(R+1); dp[0,:] = np.arange(H+1)
    for i in range(1,R+1):
        for j in range(1,H+1):
            c = 0 if ref[i-1]==hyp[j-1] else 1
            dp[i,j] = min(dp[i-1,j]+1, dp[i,j-1]+1, dp[i-1,j-1]+c)
    i,j = R,H; S=D=I=0
    while i>0 or j>0:
        if i>0 and j>0:
            c = 0 if ref[i-1]==hyp[j-1] else 1
            if dp[i,j]==dp[i-1,j-1]+c:
                if c: S+=1
                i-=1; j-=1
            elif dp[i,j]==dp[i-1,j]+1: D+=1; i-=1
            else: I+=1; j-=1
        elif i>0: D+=1; i-=1
        else: I+=1; j-=1
    return min((S+D+I)/max(R,1),1.0), S, D, I, max(R,1)


# ─────────────────────────────────────────────────────────────
#  EVALUASI
# ─────────────────────────────────────────────────────────────

def evaluate(model, eval_ids, sequences, labels, id2word, word_refs):
    model.training = False
    print(f"\n  Mendekode {len(eval_ids)} audio ...")
    wer_rows=[]; acc_rows=[]
    tS=tD=tI=tN=tC=tF=0

    for aid in eval_ids:
        X = sequences[aid]; y = labels[aid]; T = len(X)
        p = model.predict_proba(X)
        pr = np.argmax(p, axis=1)
        c  = int((pr==y).sum()); tC+=c; tF+=T
        hw = decode_to_words(p, id2word)
        rw = word_refs.get(aid, [])
        wv,S,D,I,N = levenshtein(rw, hw)
        tS+=S; tD+=D; tI+=I; tN+=N
        wer_rows.append({"audio_id":aid,"WER":round(wv*100,4),
                         "S":S,"D":D,"I":I,"N":N,
                         "referensi":" ".join(rw),
                         "hipotesis":" ".join(hw)})
        acc_rows.append({"audio_id":aid,
                         "akurasi_pct":round(c/T*100,4),
                         "benar":c,"total_frame":T})

    wc = (tS+tD+tI)/max(tN,1)*100
    ag = tC/max(tF,1)*100
    wl = [r["WER"] for r in wer_rows]
    al = [r["akurasi_pct"] for r in acc_rows]
    s  = {"n_audio":len(eval_ids),
          "wer_corpus":round(wc,4),
          "wer_mean":round(float(np.mean(wl)),4),
          "wer_min":round(float(np.min(wl)),4),
          "wer_max":round(float(np.max(wl)),4),
          "acc_global":round(ag,4),
          "acc_mean":round(float(np.mean(al)),4),
          "acc_min":round(float(np.min(al)),4),
          "acc_max":round(float(np.max(al)),4),
          "total_S":tS,"total_D":tD,"total_I":tI,"total_N":tN,
          "total_frame":tF,"total_benar":tC}

    print(f"\n── Statistik Evaluasi TDNN ────────────────────────────────")
    print(f"  Jumlah audio     : {s['n_audio']}")
    print(f"  ── WER ──────────────────────────────────────────────────")
    print(f"  WER Corpus       : {s['wer_corpus']:.2f}%")
    print(f"  WER Mean         : {s['wer_mean']:.2f}%")
    print(f"  WER Min          : {s['wer_min']:.2f}%")
    print(f"  WER Max          : {s['wer_max']:.2f}%")
    print(f"  Total S          : {s['total_S']}")
    print(f"  Total D          : {s['total_D']}")
    print(f"  Total I          : {s['total_I']}")
    print(f"  Total N (kata)   : {s['total_N']}")
    print(f"  ── Akurasi Frame ────────────────────────────────────────")
    print(f"  Akurasi Global   : {s['acc_global']:.2f}%")
    print(f"  Akurasi Mean     : {s['acc_mean']:.2f}%")
    print(f"  Akurasi Min      : {s['acc_min']:.2f}%")
    print(f"  Akurasi Max      : {s['acc_max']:.2f}%")
    print(f"  Total frame      : {s['total_frame']:,}")
    print(f"  Frame benar      : {s['total_benar']:,}")
    print(f"────────────────────────────────────────────────────────────")
    return wer_rows, acc_rows, s


# ─────────────────────────────────────────────────────────────
#  PERBANDINGAN GMM vs TDNN
# ─────────────────────────────────────────────────────────────

def print_comparison(s):
    g = {"wer_corpus":99.84,"wer_mean":99.82,"wer_min":50.00,"wer_max":100.00,
         "total_S":1730,"total_D":10394,"total_I":0,"total_N":12143}
    print(f"\n{'='*62}")
    print(f"  PERBANDINGAN GMM vs TDNN")
    print(f"{'='*62}")
    print(f"  {'Metrik':<30} {'GMM':>10} {'TDNN':>10}")
    print(f"  {'-'*52}")
    for lbl, gv, tv in [
        ("WER Corpus (%)",       f"{g['wer_corpus']:.2f}", f"{s['wer_corpus']:.2f}"),
        ("WER Mean (%)",         f"{g['wer_mean']:.2f}",   f"{s['wer_mean']:.2f}"),
        ("WER Min (%)",          f"{g['wer_min']:.2f}",    f"{s['wer_min']:.2f}"),
        ("WER Max (%)",          f"{g['wer_max']:.2f}",    f"{s['wer_max']:.2f}"),
        ("Total S (substitusi)", str(g['total_S']),        str(s['total_S'])),
        ("Total D (deletion)",   str(g['total_D']),        str(s['total_D'])),
        ("Total I (insertion)",  str(g['total_I']),        str(s['total_I'])),
        ("Total N (kata ref)",   str(g['total_N']),        str(s['total_N'])),
        ("Akurasi Frame (%)",    "N/A",                    f"{s['acc_global']:.2f}"),
    ]:
        print(f"  {lbl:<30} {gv:>10} {tv:>10}")
    print(f"{'='*62}")
    d = g['wer_corpus'] - s['wer_corpus']
    if d > 0:   print(f"  ✓ TDNN lebih baik {d:.2f}%")
    elif d < 0: print(f"  ✗ TDNN lebih buruk {abs(d):.2f}%")
    else:       print(f"  = WER sama")


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    np.random.seed(RANDOM_SEED)

    print("="*60)
    print("  TAHAP 1 — Memuat Data & Ekstraksi MFCC dari WAV")
    print("="*60)
    print(f"  Folder audio : {SAMPLED_ROOT}")
    print(f"  CSV          : {CSV_PATH}")
    print(f"  Output       : {DIR_SAVE}")
    print(f"  Label        : word-level, uniform segmentation")

    sequences, labels, word2id, id2word, word_refs, actual_dim = load_data(CSV_PATH)
    all_ids = list(sequences.keys())
    train_ids, val_ids, test_ids = split_dataset(all_ids)

    print("\n"+"="*60)
    print("  TAHAP 2 — Bangun & Latih Model TDNN")
    print("="*60)
    print(f"  Input dim  : {actual_dim}")
    for i,(h,ctx) in enumerate(zip(HIDDEN_DIMS, TDNN_CONTEXTS)):
        print(f"  Layer {i+1}     : {h} unit  context {ctx}")
    print(f"  Output dim : {len(word2id)}  (vocab kata)")
    print(f"  Dropout    : {DROPOUT_RATE} | LR: {LEARNING_RATE} | GradClip: {GRAD_CLIP}")

    model = TDNNModel(
        input_dim=actual_dim, hidden_dims=HIDDEN_DIMS,
        contexts=TDNN_CONTEXTS, output_dim=len(word2id),
        dropout=DROPOUT_RATE, lr=LEARNING_RATE,
    )

    history = train(model, train_ids, val_ids, sequences, labels)
    pd.DataFrame(history).to_csv(OUT_HISTORY_CSV, index=False)
    print(f"  {OUT_HISTORY_CSV} tersimpan")

    if os.path.exists(MODEL_PATH):
        model.load(MODEL_PATH)

    print("\n"+"="*60)
    print("  TAHAP 3 — Evaluasi WER + Akurasi (Val + Test)")
    print("="*60)
    eval_ids = val_ids + test_ids
    wr, ar, summary = evaluate(model, eval_ids, sequences, labels, id2word, word_refs)

    df_w = pd.DataFrame(wr)
    df_w[["audio_id","WER","S","D","I","N"]].to_csv(OUT_WER_CSV, index=False)
    df_w[["audio_id","referensi","hipotesis","WER"]].to_csv(OUT_DEC_CSV, index=False)
    pd.DataFrame(ar).to_csv(OUT_ACC_CSV, index=False)
    print(f"  {OUT_WER_CSV}\n  {OUT_DEC_CSV}\n  {OUT_ACC_CSV}")

    print("\n"+"="*60)
    print("  TAHAP 4 — Perbandingan GMM vs TDNN")
    print("="*60)
    print_comparison(summary)

    return model, summary, history


if __name__ == "__main__":
    model, summary, history = main()

In [3]:
"""
============================================================
  TDNN + CTC v2 — Javanese ASR
  Perbandingan dengan GMM pipeline
============================================================
  Label    : Sequence karakter per audio (CTC)
  Input    : .wav dari SAMPLED_ROOT + dataSampledSmall.csv
  Output   : WER (identik format GMM)

  CHANGELOG dari v1:
  ─────────────────────────────────────────────────────────
  BUG 1 FIXED — Gradient CTC explode
    Sebelumnya : grad = (alpha+beta) / probs  → bisa >1000
    Sekarang   : grad terhadap LOGITS langsung
                 = softmax[t,k] - (1/T*Z) * Σ exp(α+β)
                 → range terkontrol [0.01–0.3]

  BUG 2 FIXED — Numerik alpha/beta tidak stabil
    Sebelumnya : manual _log_add() rekursif
    Sekarang   : scipy.special.logsumexp per step

  BUG 3 FIXED — Tidak ada gradient clip pada output CTC
    Sebelumnya : clip hanya di dalam TDNNLayer.backward()
    Sekarang   : clip global dlog sebelum masuk backprop

  BUG 4 FIXED — Forward kembalikan log_softmax
    Sebelumnya : model.forward() → log_softmax
    Sekarang   : model.forward() → logits + softmax terpisah
                 CTC butuh log_probs = log(softmax), bukan
                 log_softmax(logits) yang sama numeriknya
                 tapi gradient pathnya berbeda
  ─────────────────────────────────────────────────────────
"""

import os, time, warnings
import numpy as np
import pandas as pd
from scipy.special import logsumexp
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────
#  KONFIGURASI
# ─────────────────────────────────────────────────────────────

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/TDNN/out/test_ctc_v4"
)
os.makedirs(DIR_SAVE, exist_ok=True)

OUT_WER_CSV     = os.path.join(DIR_SAVE, "HasilWER_TDNN_CTC.csv")
OUT_DEC_CSV     = os.path.join(DIR_SAVE, "HasilDekode_TDNN_CTC.csv")
OUT_HISTORY_CSV = os.path.join(DIR_SAVE, "HasilTraining_TDNN_CTC.csv")
MODEL_PATH      = os.path.join(DIR_SAVE, "tdnn_ctc_best.npy")

COL_PATH_DATA = "Path_Data"
COL_SENTENCES = "sentences"

# ── MFCC ──
SR          = 16000
N_MFCC      = 13
N_FFT       = 512
HOP_LENGTH  = 160
WIN_LENGTH  = 400
N_MELS      = 26
PRE_EMPH    = 0.97
FMIN        = 0.0
FMAX        = 4000.0
DELTA_WIDTH = 2
INPUT_DIM   = N_MFCC * 3   # 39

# ── Arsitektur ──
TDNN_CONTEXTS = [[-2, 2], [-2, 0], [-3, 3], [0, 0]]
HIDDEN_DIMS   = [512, 512, 512, 512]
DROPOUT_RATE  = 0.2
LEARNING_RATE = 1e-4      # FIX 5: lebih konservatif dari 3e-4
GRAD_CLIP     = 1.0       # FIX 7: clip global, lebih ketat

# ── Training ──
BATCH_SIZE  = 16          # utterance per update
MAX_EPOCHS  = 50
EARLY_STOP  = 7

# ── Split ──
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
RANDOM_SEED = 42

BLANK = 0   # index blank selalu 0


# ─────────────────────────────────────────────────────────────
#  EKSTRAKSI MFCC
# ─────────────────────────────────────────────────────────────

try:
    import librosa
    LIBROSA_OK = True
except ImportError:
    LIBROSA_OK = False
try:
    from scipy.io import wavfile
    SCIPY_OK = True
except ImportError:
    SCIPY_OK = False


def extract_features(wav_path: str) -> np.ndarray:
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(wav_path)
    return _librosa(wav_path) if LIBROSA_OK else _manual(wav_path)


def _librosa(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    if len(y) == 0:
        raise ValueError("Audio kosong")
    y    = np.append(y[0], y[1:] - PRE_EMPH * y[:-1])
    mfcc = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT,
        hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, window="hamming"
    )
    d1   = librosa.feature.delta(mfcc, width=DELTA_WIDTH * 2 + 1)
    d2   = librosa.feature.delta(mfcc, order=2, width=DELTA_WIDTH * 2 + 1)
    feat = np.vstack([mfcc, d1, d2]).T.astype(np.float32)
    # Normalisasi per-utterance (mean-variance)
    feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
    return feat


def _delta_vec(feat, width=2):
    T     = feat.shape[0]
    denom = 2.0 * sum(n * n for n in range(1, width + 1))
    delta = np.zeros_like(feat)
    for n in range(1, width + 1):
        fwd   = feat[np.clip(np.arange(T) + n, 0, T - 1)]
        bwd   = feat[np.clip(np.arange(T) - n, 0, T - 1)]
        delta += n * (fwd - bwd)
    return delta / denom


def _manual(wav_path):
    if not SCIPY_OK:
        raise ImportError("Butuh librosa atau scipy")
    sr, audio = wavfile.read(wav_path)
    if audio.dtype == np.int16:
        audio = audio / 32768.0
    elif audio.dtype == np.int32:
        audio = audio / 2147483648.0
    audio = audio.astype(np.float32)
    if sr != SR:
        n     = int(len(audio) * SR / sr)
        audio = np.interp(np.linspace(0, len(audio) - 1, n),
                          np.arange(len(audio)), audio)
    y    = np.append(audio[0], audio[1:] - PRE_EMPH * audio[:-1])
    T_fr = 1 + (len(y) - WIN_LENGTH) // HOP_LENGTH
    idx  = (np.arange(WIN_LENGTH)[None, :] +
            np.arange(T_fr)[:, None] * HOP_LENGTH)
    frames  = y[idx] * np.hamming(WIN_LENGTH)
    fft_m   = np.abs(np.fft.rfft(frames, n=N_FFT))
    def hz2mel(f): return 1127 * np.log1p(f / 700)
    def mel2hz(m): return 700 * (np.exp(m / 1127) - 1)
    pts  = mel2hz(np.linspace(hz2mel(FMIN), hz2mel(FMAX), N_MELS + 2))
    bins = np.floor((N_FFT + 1) * pts / SR).astype(int)
    fb   = np.zeros((N_MELS, N_FFT // 2 + 1))
    for m in range(1, N_MELS + 1):
        for k in range(bins[m - 1], bins[m]):
            fb[m-1, k] = (k - bins[m-1]) / (bins[m] - bins[m-1] + 1e-10)
        for k in range(bins[m], bins[m + 1]):
            fb[m-1, k] = (bins[m+1] - k) / (bins[m+1] - bins[m] + 1e-10)
    log_mel = np.log10(np.maximum(fft_m ** 2 @ fb.T, 1e-10))
    N_      = N_MELS
    ni      = np.arange(N_MFCC)[:, None]
    ki      = np.arange(N_)[None, :] + 0.5
    dct_m   = np.cos(np.pi * ni * ki / N_) * np.sqrt(2 / N_)
    mfcc    = log_mel @ dct_m.T
    d1      = _delta_vec(mfcc, DELTA_WIDTH)
    d2      = _delta_vec(d1, DELTA_WIDTH)
    feat    = np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)
    feat    = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
    return feat


# ─────────────────────────────────────────────────────────────
#  VOCAB KARAKTER
# ─────────────────────────────────────────────────────────────

def build_char_vocab(sentences: list) -> tuple:
    chars   = set()
    for s in sentences:
        chars.update(s.lower().strip())
    vocab   = ["<blank>"] + sorted(chars)
    char2id = {c: i for i, c in enumerate(vocab)}
    id2char = {i: c for c, i in char2id.items()}
    return vocab, char2id, id2char


def sentence_to_ids(sentence: str, char2id: dict) -> list:
    return [char2id[c] for c in sentence.lower().strip()
            if c in char2id]


# ─────────────────────────────────────────────────────────────
#  DATA LOADER
# ─────────────────────────────────────────────────────────────

def find_wav(audio_id: str) -> str:
    for root, _, files in os.walk(SAMPLED_ROOT):
        for f in files:
            if (os.path.splitext(f)[0] == audio_id
                    and f.lower().endswith(".wav")):
                return os.path.join(root, f)
    return ""


def load_data(csv_path: str) -> tuple:
    print(f"  Membaca {csv_path} ...")
    df = pd.read_csv(csv_path)
    print(f"  Shape CSV : {df.shape}")
    print(f"  Kolom     : {list(df.columns)}")

    for col in [COL_PATH_DATA, COL_SENTENCES]:
        if col not in df.columns:
            raise ValueError(f"Kolom '{col}' tidak ada.")

    sentences = df[COL_SENTENCES].astype(str).str.lower().str.strip().tolist()
    vocab, char2id, id2char = build_char_vocab(sentences)
    print(f"  Vocab karakter   : {len(vocab)-1}  (+1 <blank> → total {len(vocab)})")

    sequences, targets, word_refs = {}, {}, {}
    skipped = 0

    print(f"  Mengekstrak MFCC dari {len(df)} file .wav ...")
    for idx, row in df.iterrows():
        audio_id = os.path.splitext(
            os.path.basename(str(row[COL_PATH_DATA])))[0]
        kalimat  = str(row[COL_SENTENCES]).lower().strip()
        words    = kalimat.split()

        wav_path = str(row[COL_PATH_DATA])
        if not os.path.isfile(wav_path):
            wav_path = find_wav(audio_id)
        if not wav_path:
            skipped += 1
            continue

        try:
            feats  = extract_features(wav_path)
            T      = feats.shape[0]
            target = sentence_to_ids(kalimat, char2id)

            if T == 0 or len(target) == 0:
                skipped += 1; continue

            # CTC constraint: T >= 2*L+1 (minimal frame untuk extended label)
            if T < 2 * len(target) + 1:
                skipped += 1
                continue

            sequences[audio_id] = feats
            targets[audio_id]   = target
            word_refs[audio_id] = words

        except Exception as e:
            print(f"  ⚠ {audio_id}: {e}")
            skipped += 1
            continue

        if (idx + 1) % 200 == 0 or (idx + 1) == len(df):
            print(f"    [{idx+1}/{len(df)}] diproses, {skipped} dilewati")

    total_frames = sum(v.shape[0] for v in sequences.values())
    actual_dim   = next(iter(sequences.values())).shape[1]

    print(f"\n  Audio berhasil : {len(sequences)}")
    print(f"  Audio dilewati : {skipped}")
    print(f"  Total frame    : {total_frames:,}")
    print(f"  Input dim      : {actual_dim}")
    print(f"  Output dim     : {len(vocab)}  ← karakter + blank (CTC)")

    return sequences, targets, word_refs, char2id, id2char, vocab, actual_dim


# ─────────────────────────────────────────────────────────────
#  AKTIVASI
# ─────────────────────────────────────────────────────────────

def relu(x):      return np.maximum(0.0, x)
def relu_grad(x): return (x > 0).astype(np.float32)

def softmax_stable(x):
    """Softmax numerically stable. x: (T, V)"""
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / (e.sum(axis=1, keepdims=True) + 1e-30)


# ─────────────────────────────────────────────────────────────
#  CTC LOSS + GRADIENT TERHADAP LOGITS
#
#  FIX UTAMA:
#  Gradient dihitung terhadap LOGITS (sebelum softmax), bukan
#  terhadap log_probs. Formula:
#
#    dL/dlogit[t,k] = probs[t,k]
#                   - (1 / (T * Z)) * Σ_{s: l'[s]=k} exp(α[t,s]+β[t,s])
#
#  di mana Z = exp(log_likelihood).
#  Ini menghindari pembagian dengan probs yang bisa → 0.
#
#  alpha/beta dihitung dengan scipy logsumexp agar stabil
#  secara numerik meski T besar (T~570 frame).
# ─────────────────────────────────────────────────────────────

def ctc_loss_and_grad(logits: np.ndarray,
                      targets: list) -> tuple:
    """
    logits  : (T, V)  output mentah TDNN sebelum softmax
    targets : list[int]  sequence label tanpa blank
    return  : loss (scalar), grad_logits (T, V)
    """
    T, V = logits.shape
    L    = len(targets)

    # Softmax dari logits
    probs     = softmax_stable(logits)                   # (T, V)
    log_probs = np.log(probs + 1e-30).astype(np.float64) # (T, V)

    # Extended label: [blank, t1, blank, t2, ..., tL, blank]
    l_ = []
    for t in targets:
        l_.append(BLANK)
        l_.append(t)
    l_.append(BLANK)
    S = len(l_)   # 2L+1

    NEG_INF = -np.inf

    # ── Forward (alpha) dengan logsumexp ─────────────────────
    alpha = np.full((T, S), NEG_INF, dtype=np.float64)
    alpha[0, 0] = log_probs[0, BLANK]
    if S > 1:
        alpha[0, 1] = log_probs[0, l_[1]]

    for t in range(1, T):
        for s in range(S):
            c     = l_[s]
            cands = [alpha[t-1, s]]
            if s > 0:
                cands.append(alpha[t-1, s-1])
            if s > 1 and c != BLANK and c != l_[s-2]:
                cands.append(alpha[t-1, s-2])
            # logsumexp: stabil meski nilai sangat negatif
            lse = logsumexp(cands)
            alpha[t, s] = (lse + log_probs[t, c]
                           if np.isfinite(lse) else NEG_INF)

    # ── Backward (beta) dengan logsumexp ─────────────────────
    beta = np.full((T, S), NEG_INF, dtype=np.float64)
    beta[T-1, S-1] = 0.0
    if S > 1:
        beta[T-1, S-2] = 0.0

    for t in range(T-2, -1, -1):
        for s in range(S):
            c     = l_[s]
            cands = [beta[t+1, s] + log_probs[t+1, c]]
            if s < S-1:
                cands.append(beta[t+1, s+1] + log_probs[t+1, l_[s+1]])
            if (s < S-2 and l_[s+2] != BLANK and l_[s+2] != c):
                cands.append(beta[t+1, s+2] + log_probs[t+1, l_[s+2]])
            lse = logsumexp(cands)
            beta[t, s] = lse if np.isfinite(lse) else NEG_INF

    # ── Log-likelihood ───────────────────────────────────────
    finals = [alpha[T-1, S-1]]
    if S > 1:
        finals.append(alpha[T-1, S-2])
    ll = logsumexp(finals)

    if not np.isfinite(ll):
        # Sequence tidak dapat direpresentasikan → skip, grad nol
        return 0.0, np.zeros((T, V), dtype=np.float32)

    loss = float(-ll / T)

    # ── Gradient terhadap LOGITS (FIX BUG 1) ─────────────────
    # dL/dlogit[t,k] = probs[t,k] - term_k[t]
    # term_k[t] = (1/(T*Z)) * Σ_{s:l'[s]=k} exp(α[t,s]+β[t,s])
    Z         = np.exp(ll)
    grad_log  = probs.copy().astype(np.float64)   # probs bagian softmax
    ab        = alpha + beta                       # (T, S) log-domain

    for s in range(S):
        c     = l_[s]
        valid = np.isfinite(ab[:, s])
        if valid.any():
            grad_log[valid, c] -= (
                np.exp(ab[valid, s]) / (Z * T + 1e-30)
            )

    return loss, grad_log.astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  TDNN LAYER
#  Perubahan: clip global di model, bukan per layer
#  (layer tidak clip sendiri — agar gradient flow tidak terpotong
#   sebelum dikumpulkan dan dinormalisasi)
# ─────────────────────────────────────────────────────────────

class TDNNLayer:
    def __init__(self, in_dim, out_dim, context, lr):
        self.context = context
        self.ctx_len = context[1] - context[0] + 1
        self.lr      = lr
        fan_in       = in_dim * self.ctx_len
        self.W = (np.random.randn(fan_in, out_dim) *
                  np.sqrt(2.0 / fan_in)).astype(np.float32)
        self.b = np.zeros(out_dim, dtype=np.float32)
        self.mW = np.zeros_like(self.W)
        self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b)
        self.vb = np.zeros_like(self.b)
        self.t  = 0
        self._X = None

    def splice(self, X):
        T   = X.shape[0]
        idx = np.arange(T)
        return np.concatenate(
            [X[np.clip(idx + off, 0, T - 1)]
             for off in range(self.context[0], self.context[1] + 1)],
            axis=1
        )

    def forward(self, Xsp):
        self._X = Xsp
        return Xsp @ self.W + self.b

    def backward(self, dout):
        dW   = self._X.T @ dout / len(dout)
        db   = dout.mean(0)
        dXsp = dout @ self.W.T
        self._adam(dW, db)
        B, D_tot = dXsp.shape
        D  = D_tot // self.ctx_len
        dX = np.zeros((B, D), dtype=np.float32)
        for k in range(self.ctx_len):
            dX += dXsp[:, k * D:(k + 1) * D]
        return dX

    def _adam(self, dW, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t  += 1
        self.mW  = b1 * self.mW + (1 - b1) * dW
        self.vW  = b2 * self.vW + (1 - b2) * dW ** 2
        self.W  -= (self.lr * (self.mW / (1 - b1 ** self.t))
                    / (np.sqrt(self.vW / (1 - b2 ** self.t)) + eps))
        self.mb  = b1 * self.mb + (1 - b1) * db
        self.vb  = b2 * self.vb + (1 - b2) * db ** 2
        self.b  -= (self.lr * (self.mb / (1 - b1 ** self.t))
                    / (np.sqrt(self.vb / (1 - b2 ** self.t)) + eps))


# ─────────────────────────────────────────────────────────────
#  MODEL TDNN
#  FIX 4: forward() kembalikan logits mentah (bukan log_softmax)
#  CTC butuh logits untuk menghitung gradient yang benar
# ─────────────────────────────────────────────────────────────

class TDNNModel:
    def __init__(self, input_dim, hidden_dims, contexts,
                 output_dim, dropout=0.2, lr=1e-4):
        assert len(hidden_dims) == len(contexts)
        self.dropout  = dropout
        self.training = True
        self.layers   = []
        in_d = input_dim
        for h, ctx in zip(hidden_dims, contexts):
            self.layers.append(TDNNLayer(in_d, h, ctx, lr))
            in_d = h
        self.layers.append(TDNNLayer(in_d, output_dim, [0, 0], lr))
        self._rc = []
        self._dc = []

    def forward(self, X) -> np.ndarray:
        """Kembalikan LOGITS (T, V) — belum softmax."""
        self._rc = []
        self._dc = []
        h = X
        for layer in self.layers[:-1]:
            z = layer.forward(layer.splice(h))
            a = relu(z)
            if self.training and self.dropout > 0:
                mask = (np.random.rand(*a.shape) > self.dropout
                        ).astype(np.float32)
                a    = a * mask / (1.0 - self.dropout)
            else:
                mask = np.ones(a.shape, dtype=np.float32)
            self._rc.append(z)
            self._dc.append(mask)
            h = a
        out = self.layers[-1]
        return out.forward(out.splice(h))   # raw logits (T, V)

    def ctc_step(self, X, target):
        """
        Forward + CTC loss + backward untuk 1 utterance.
        Return loss scalar.
        """
        logits = self.forward(X)                      # (T, V)
        loss, dlogit = ctc_loss_and_grad(logits, target)

        if loss == 0.0:   # utterance tidak valid
            return 0.0

        # FIX 3: clip gradient global sebelum masuk TDNN backward
        gnorm = np.sqrt((dlogit ** 2).sum())
        if gnorm > GRAD_CLIP:
            dlogit = dlogit * (GRAD_CLIP / gnorm)

        # Backprop melalui output layer, lalu hidden layers
        sc = 1.0 / (1.0 - self.dropout + 1e-9) if self.dropout > 0 else 1.0
        dh = self.layers[-1].backward(dlogit)
        for i in reversed(range(len(self.layers) - 1)):
            dh = dh * self._dc[i] * sc
            dh = dh * relu_grad(self._rc[i])
            dh = self.layers[i].backward(dh)
        return loss

    def get_log_probs(self, X) -> np.ndarray:
        """Untuk dekode: kembalikan log_softmax (T, V)."""
        self.training = False
        logits    = self.forward(X)
        probs     = softmax_stable(logits)
        return np.log(probs + 1e-30)

    def save(self, path):
        np.save(path,
                {"w": [(l.W.copy(), l.b.copy()) for l in self.layers]},
                allow_pickle=True)

    def load(self, path):
        d = np.load(path, allow_pickle=True).item()
        for i, (W, b) in enumerate(d["w"]):
            self.layers[i].W = W
            self.layers[i].b = b
        print(f"  Model dimuat: {path}")


# ─────────────────────────────────────────────────────────────
#  SPLIT
# ─────────────────────────────────────────────────────────────

def split_dataset(ids):
    rng  = np.random.default_rng(RANDOM_SEED)
    ids  = list(ids)
    perm = rng.permutation(len(ids))
    n    = len(ids)
    ntr  = int(n * TRAIN_RATIO)
    nva  = int(n * VAL_RATIO)
    tr   = [ids[i] for i in perm[:ntr]]
    va   = [ids[i] for i in perm[ntr:ntr + nva]]
    te   = [ids[i] for i in perm[ntr + nva:]]
    print(f"  Split → train:{len(tr)}  val:{len(va)}  test:{len(te)}")
    return tr, va, te


# ─────────────────────────────────────────────────────────────
#  TRAINING
#  FIX 6: Akumulasi loss per batch, weight update per batch
#          (bukan per utterance)
#  Ini mengurangi noise update dan menstabilkan learning
# ─────────────────────────────────────────────────────────────

def train(model, train_ids, val_ids, sequences, targets):
    rng  = np.random.default_rng(RANDOM_SEED)
    hist = []
    best = float("inf")
    pat  = 0

    print(f"\n  Train: {len(train_ids)} audio | Val: {len(val_ids)} audio")
    print(f"  Epochs: {MAX_EPOCHS} | EarlyStop: {EARLY_STOP} | "
          f"Batch: {BATCH_SIZE} utterance")
    print(f"\n  {'Ep':>4}  {'TrLoss':>9}  {'VaLoss':>9}  {'t':>7}")
    print(f"  {'-'*40}")

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        model.training = True

        perm  = rng.permutation(len(train_ids))
        order = [train_ids[i] for i in perm]

        total_loss = 0.0
        n_batches  = 0

        for start in range(0, len(order), BATCH_SIZE):
            batch = order[start:start + BATCH_SIZE]
            batch_loss = 0.0
            valid      = 0
            for aid in batch:
                try:
                    lv = model.ctc_step(sequences[aid], targets[aid])
                    if lv > 0:
                        batch_loss += lv
                        valid      += 1
                except Exception:
                    continue
            if valid > 0:
                total_loss += batch_loss / valid
                n_batches  += 1

        tr_loss = total_loss / max(n_batches, 1)

        # ── Validasi ─────────────────────────────────────────
        model.training = False
        val_loss = 0.0
        val_n    = 0
        for aid in val_ids:
            try:
                logits = model.forward(sequences[aid])
                lv, _  = ctc_loss_and_grad(logits, targets[aid])
                if lv > 0:
                    val_loss += lv
                    val_n    += 1
            except Exception:
                continue
        val_loss /= max(val_n, 1)

        elapsed = time.time() - t0
        hist.append({
            "epoch":      epoch,
            "train_loss": round(tr_loss,  6),
            "val_loss":   round(val_loss, 6),
        })
        print(f"  {epoch:4d}  {tr_loss:9.4f}  {val_loss:9.4f}  {elapsed:6.1f}s")

        if val_loss < best - 1e-5:
            best = val_loss
            pat  = 0
            model.save(MODEL_PATH)
        else:
            pat += 1
            if pat >= EARLY_STOP:
                print(f"\n  ✓ Early stop epoch {epoch} "
                      f"(best val_loss={best:.4f})")
                break

    return hist


# ─────────────────────────────────────────────────────────────
#  GREEDY CTC DECODE
# ─────────────────────────────────────────────────────────────

def ctc_greedy_decode(log_probs: np.ndarray, id2char: dict) -> str:
    preds = np.argmax(log_probs, axis=1)
    chars = []
    prev  = None
    for p in preds:
        if p != prev:
            if p != BLANK:
                chars.append(id2char.get(int(p), ""))
            prev = p
    return "".join(chars)


# ─────────────────────────────────────────────────────────────
#  WER
# ─────────────────────────────────────────────────────────────

def levenshtein(ref, hyp):
    R, H = len(ref), len(hyp)
    dp   = np.zeros((R + 1, H + 1), dtype=int)
    dp[:, 0] = np.arange(R + 1)
    dp[0, :] = np.arange(H + 1)
    for i in range(1, R + 1):
        for j in range(1, H + 1):
            c       = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i,j] = min(dp[i-1,j]+1, dp[i,j-1]+1, dp[i-1,j-1]+c)
    i, j  = R, H
    S=D=I = 0
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            c = 0 if ref[i-1] == hyp[j-1] else 1
            if dp[i,j] == dp[i-1,j-1] + c:
                if c: S += 1
                i -= 1; j -= 1
            elif dp[i,j] == dp[i-1,j] + 1:
                D += 1; i -= 1
            else:
                I += 1; j -= 1
        elif i > 0:
            D += 1; i -= 1
        else:
            I += 1; j -= 1
    return min((S+D+I)/max(R,1), 1.0), S, D, I, max(R, 1)


# ─────────────────────────────────────────────────────────────
#  EVALUASI
# ─────────────────────────────────────────────────────────────

def evaluate(model, eval_ids, sequences, word_refs, id2char):
    model.training = False
    print(f"\n  Mendekode {len(eval_ids)} audio ...")
    wer_rows = []
    tS=tD=tI=tN = 0

    for aid in eval_ids:
        lp = model.get_log_probs(sequences[aid])
        decoded = ctc_greedy_decode(lp, id2char)
        hw = decoded.split()
        rw = word_refs.get(aid, [])
        wv, S, D, I, N = levenshtein(rw, hw)
        tS+=S; tD+=D; tI+=I; tN+=N
        wer_rows.append({
            "audio_id":  aid,
            "WER":       round(wv * 100, 4),
            "S": S, "D": D, "I": I, "N": N,
            "referensi": " ".join(rw),
            "hipotesis": " ".join(hw),
        })

    wc = (tS+tD+tI) / max(tN,1) * 100
    wl = [r["WER"] for r in wer_rows]
    s  = {
        "n_audio":    len(eval_ids),
        "wer_corpus": round(wc, 4),
        "wer_mean":   round(float(np.mean(wl)), 4),
        "wer_min":    round(float(np.min(wl)),  4),
        "wer_max":    round(float(np.max(wl)),  4),
        "total_S": tS, "total_D": tD, "total_I": tI, "total_N": tN,
    }

    print(f"\n── Statistik Evaluasi TDNN-CTC ─────────────────────────────")
    print(f"  Jumlah audio   : {s['n_audio']}")
    print(f"  WER Corpus     : {s['wer_corpus']:.2f}%")
    print(f"  WER Mean       : {s['wer_mean']:.2f}%")
    print(f"  WER Min        : {s['wer_min']:.2f}%")
    print(f"  WER Max        : {s['wer_max']:.2f}%")
    print(f"  Total S        : {s['total_S']}")
    print(f"  Total D        : {s['total_D']}")
    print(f"  Total I        : {s['total_I']}")
    print(f"  Total N (kata) : {s['total_N']}")
    print(f"─────────────────────────────────────────────────────────────")
    return wer_rows, s


# ─────────────────────────────────────────────────────────────
#  PERBANDINGAN GMM vs TDNN-CTC
# ─────────────────────────────────────────────────────────────

def print_comparison(s):
    g = {
        "wer_corpus": 99.84, "wer_mean": 99.82,
        "wer_min":    50.00, "wer_max": 100.00,
        "total_S": 1730, "total_D": 10394,
        "total_I": 0,    "total_N": 12143,
    }
    print(f"\n{'='*62}")
    print(f"  PERBANDINGAN GMM vs TDNN-CTC v2")
    print(f"{'='*62}")
    print(f"  {'Metrik':<30} {'GMM':>10} {'TDNN-CTC':>10}")
    print(f"  {'-'*52}")
    for lbl, gv, tv in [
        ("WER Corpus (%)",       f"{g['wer_corpus']:.2f}", f"{s['wer_corpus']:.2f}"),
        ("WER Mean (%)",         f"{g['wer_mean']:.2f}",   f"{s['wer_mean']:.2f}"),
        ("WER Min (%)",          f"{g['wer_min']:.2f}",    f"{s['wer_min']:.2f}"),
        ("WER Max (%)",          f"{g['wer_max']:.2f}",    f"{s['wer_max']:.2f}"),
        ("Total S (substitusi)", str(g['total_S']),        str(s['total_S'])),
        ("Total D (deletion)",   str(g['total_D']),        str(s['total_D'])),
        ("Total I (insertion)",  str(g['total_I']),        str(s['total_I'])),
        ("Total N (kata ref)",   str(g['total_N']),        str(s['total_N'])),
    ]:
        print(f"  {lbl:<30} {gv:>10} {tv:>10}")
    print(f"{'='*62}")
    d = g["wer_corpus"] - s["wer_corpus"]
    if   d > 0: print(f"  ✓ TDNN-CTC lebih baik {d:.2f}% dari GMM")
    elif d < 0: print(f"  ✗ TDNN-CTC lebih buruk {abs(d):.2f}% dari GMM")
    else:       print(f"  = WER sama")
    print(f"{'='*62}\n")


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    np.random.seed(RANDOM_SEED)

    print("=" * 60)
    print("  TAHAP 1 — Memuat Data & Ekstraksi MFCC dari WAV")
    print("=" * 60)
    print(f"  Folder audio : {SAMPLED_ROOT}")
    print(f"  CSV          : {CSV_PATH}")
    print(f"  Output       : {DIR_SAVE}")
    print(f"  Label        : sequence karakter per audio (CTC)")

    (sequences, targets, word_refs,
     char2id, id2char, vocab, actual_dim) = load_data(CSV_PATH)

    all_ids = list(sequences.keys())
    train_ids, val_ids, test_ids = split_dataset(all_ids)

    print("\n" + "=" * 60)
    print("  TAHAP 2 — Bangun & Latih Model TDNN + CTC")
    print("=" * 60)
    print(f"  Input dim  : {actual_dim}")
    for i, (h, ctx) in enumerate(zip(HIDDEN_DIMS, TDNN_CONTEXTS)):
        print(f"  Layer {i+1}     : {h} unit  context {ctx}")
    print(f"  Output dim : {len(vocab)}  (karakter + blank)")
    print(f"  Loss       : CTC — gradient terhadap logits")
    print(f"  Dropout    : {DROPOUT_RATE} | LR: {LEARNING_RATE} | "
          f"GradClip: {GRAD_CLIP}")

    model = TDNNModel(
        input_dim=actual_dim,
        hidden_dims=HIDDEN_DIMS,
        contexts=TDNN_CONTEXTS,
        output_dim=len(vocab),
        dropout=DROPOUT_RATE,
        lr=LEARNING_RATE,
    )

    history = train(model, train_ids, val_ids, sequences, targets)
    pd.DataFrame(history).to_csv(OUT_HISTORY_CSV, index=False)
    print(f"  History tersimpan: {OUT_HISTORY_CSV}")

    if os.path.exists(MODEL_PATH):
        model.load(MODEL_PATH)

    print("\n" + "=" * 60)
    print("  TAHAP 3 — Evaluasi WER (Val + Test)")
    print("=" * 60)
    eval_ids       = val_ids + test_ids
    wer_rows, summ = evaluate(model, eval_ids, sequences,
                              word_refs, id2char)

    df_w = pd.DataFrame(wer_rows)
    df_w[["audio_id","WER","S","D","I","N"]].to_csv(
        OUT_WER_CSV, index=False)
    df_w[["audio_id","referensi","hipotesis","WER"]].to_csv(
        OUT_DEC_CSV, index=False)
    print(f"  {OUT_WER_CSV}\n  {OUT_DEC_CSV}")

    print("\n" + "=" * 60)
    print("  TAHAP 4 — Perbandingan GMM vs TDNN-CTC")
    print("=" * 60)
    print_comparison(summ)

    return model, summ, history


if __name__ == "__main__":
    model, summary, history = main()

  TAHAP 1 — Memuat Data & Ekstraksi MFCC dari WAV
  Folder audio : D:\doc\2026\FINAL_CODE\data\sampled_small
  CSV          : D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv
  Output       : D:/doc/2026/FINAL_CODE/src/codeStat/modelling/voiceModelling/srcVoiceModelling/TDNN/out/test_ctc_v4
  Label        : sequence karakter per audio (CTC)
  Membaca D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv ...
  Shape CSV : (1749, 4)
  Kolom     : ['folder', 'subfolder', 'sentences', 'Path_Data']
  Vocab karakter   : 34  (+1 <blank> → total 35)
  Mengekstrak MFCC dari 1749 file .wav ...
    [200/1749] diproses, 0 dilewati
    [400/1749] diproses, 0 dilewati
    [600/1749] diproses, 0 dilewati
    [800/1749] diproses, 0 dilewati
    [1000/1749] diproses, 0 dilewati
    [1200/1749] diproses, 0 dilewati
    [1400/1749] diproses, 0 dilewati
    [1600/1749] diproses, 0 dilew

KeyboardInterrupt: 

In [ ]:
"""
============================================================
  TDNN + CTC v5 — Javanese ASR (Phoneme-level)
  PERBAIKAN dari v4:

  [FIX 1] NORMALISASI LABEL sebelum G2P
          - Angka dikonversi ke kata Jawa (17 → pitulas)
          - Tanda baca dihapus sebelum masuk G2P
          - Mencegah karakter aneh masuk ke vocab fonem

  [FIX 2] NORMALISASI MFCC GLOBAL (bukan per-file)
          - Hitung mean/std dari seluruh training set
          - Terapkan ke semua split (train/val/test)
          - Distribusi input konsisten antar batch

  [FIX 3] BATCH NORMALIZATION pada setiap layer TDNN
          - Stabilkan aktivasi antar layer
          - Gradien lebih stabil, konvergensi lebih cepat
          - Diterapkan setelah linear, sebelum ReLU

  [FIX 4] VAL LOSS dihitung dengan training=False konsisten
          - Gunakan ctc_loss_and_grad langsung pada logits
          - Pastikan dropout non-aktif saat validasi

  [FIX 5] LOG WER PADA TRAIN SET (setiap 5 epoch)
          - Deteksi dini overfitting/underfitting
          - Sample 50 audio dari train untuk efisiensi

  [FIX 6] SIMPLE BEAM SEARCH CTC DECODE (beam=10)
          - Lebih baik dari greedy untuk sequence pendek
          - Menurunkan WER tanpa perubahan model
============================================================
"""

import os, time, warnings
import numpy as np
import pandas as pd
from scipy.special import logsumexp
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  KONFIGURASI
# ─────────────────────────────────────────────────────────────

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/GMM_TDNN/out/TDNN_v5"
)
os.makedirs(DIR_SAVE, exist_ok=True)

OUT_WER_CSV     = os.path.join(DIR_SAVE, "HasilWER_TDNN_v5.csv")
OUT_DEC_CSV     = os.path.join(DIR_SAVE, "HasilDekode_TDNN_v5.csv")
OUT_HISTORY_CSV = os.path.join(DIR_SAVE, "HasilTraining_TDNN_v5.csv")
MODEL_PATH      = os.path.join(DIR_SAVE, "tdnn_v5_best.npy")
NORM_PATH       = os.path.join(DIR_SAVE, "mfcc_global_norm.npy")

COL_PATH_DATA = "Path_Data"
COL_SENTENCES = "sentences"

# ── MFCC (identik GMM) ──
SR=16000; N_MFCC=13; N_FFT=512; HOP_LENGTH=160
WIN_LENGTH=400; N_MELS=26; PRE_EMPH=0.97
FMIN=0.0; FMAX=4000.0; DELTA_WIDTH=2
INPUT_DIM = N_MFCC * 3   # 39

# ── Arsitektur ──
TDNN_CONTEXTS = [[-2,2],[-2,0],[-3,3]]
HIDDEN_DIMS   = [256, 256, 256]
DROPOUT_RATE  = 0.2
LR_INIT       = 1e-3
LR_WARMUP     = 1e-5
GRAD_CLIP     = 1.0

# ── Training ──
BATCH_SIZE   = 16
MAX_EPOCHS   = 60
EARLY_STOP   = 12
WARMUP_EPOCH = 5
LR_DECAY     = 0.5
LR_PATIENCE  = 5
LR_MIN       = 1e-6

# ── Split ──
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
RANDOM_SEED = 42
BLANK       = 0

# ── Beam search ──
BEAM_WIDTH = 10


# ─────────────────────────────────────────────────────────────
#  [FIX 1] NORMALISASI TEKS BAHASA JAWA sebelum G2P
# ─────────────────────────────────────────────────────────────

# Kamus angka → kata Jawa (0-20, puluhan, ratusan)
_ANGKA_JAWA = {
    0:'nol', 1:'siji', 2:'loro', 3:'telu', 4:'papat', 5:'lima',
    6:'enem', 7:'pitu', 8:'wolu', 9:'songo', 10:'sepuluh',
    11:'sewelas', 12:'rolas', 13:'telulas', 14:'patbelas',
    15:'limolas', 16:'nembelas', 17:'pitulas', 18:'wolulas',
    19:'sangalas', 20:'rongpuluh',
}

def _angka_ke_jawa(n: int) -> str:
    """Konversi integer ke kata Jawa (support 0-9999)."""
    if n in _ANGKA_JAWA:
        return _ANGKA_JAWA[n]
    if n < 100:
        puluhan = n // 10
        satuan  = n % 10
        if puluhan == 1:
            pfx = "sepuluh" if satuan == 0 else _ANGKA_JAWA.get(10+satuan, f"{satuan}")
            return pfx
        p = _ANGKA_JAWA.get(puluhan, str(puluhan)) + "puluh"
        return p if satuan == 0 else p + _ANGKA_JAWA.get(satuan, str(satuan))
    if n < 1000:
        ratus = n // 100
        sisa  = n % 100
        pfx = "satus" if ratus == 1 else _ANGKA_JAWA.get(ratus, str(ratus)) + "atus"
        return pfx if sisa == 0 else pfx + _angka_ke_jawa(sisa)
    ribu = n // 1000
    sisa = n % 1000
    pfx = "sewu" if ribu == 1 else _ANGKA_JAWA.get(ribu, str(ribu)) + "ewu"
    return pfx if sisa == 0 else pfx + _angka_ke_jawa(sisa)


def normalize_text(text: str) -> str:
    """
    [FIX 1] Normalisasi teks sebelum G2P:
      1. Lowercase + strip
      2. Angka → kata Jawa
      3. Hapus tanda baca (kecuali vokal berdiakritik)
      4. Collapse whitespace
    """
    import re
    text = text.lower().strip()

    # Ganti angka dengan kata Jawa
    def replace_num(m):
        return ' ' + _angka_ke_jawa(int(m.group())) + ' '
    text = re.sub(r'\d+', replace_num, text)

    # Hapus tanda baca kecuali huruf, spasi, dan vokal berdiakritik
    text = re.sub(r"[^a-zéèêâ\s]", ' ', text)

    # Collapse spasi ganda
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# ─────────────────────────────────────────────────────────────
#  G2P BAHASA JAWA
# ─────────────────────────────────────────────────────────────

DIGRAPHS = {'ng': 'ŋ', 'ny': 'ɲ', 'th': 'tʰ', 'dh': 'dʰ'}


def javanese_g2p(text: str) -> list:
    """
    Konversi teks Bahasa Jawa (sudah dinormalisasi) → sequence fonem.
    """
    text     = normalize_text(text)   # [FIX 1] normalisasi dulu
    phonemes = []
    i        = 0

    while i < len(text):
        ch = text[i]

        if ch == ' ':
            if phonemes and phonemes[-1] != '<sp>':
                phonemes.append('<sp>')
            i += 1
            continue

        if not ch.isalpha() and ch not in 'éèâê':
            i += 1
            continue

        if i + 1 < len(text):
            two = text[i:i+2]
            if two in DIGRAPHS:
                phonemes.append(DIGRAPHS[two])
                i += 2
                continue

        phonemes.append(ch)
        i += 1

    return phonemes


def build_phoneme_vocab(sentences: list) -> tuple:
    all_phn = set()
    for s in sentences:
        all_phn.update(javanese_g2p(s))
    vocab   = ['<blank>'] + sorted(all_phn)
    phn2id  = {p: i for i, p in enumerate(vocab)}
    id2phn  = {i: p for p, i in phn2id.items()}
    return vocab, phn2id, id2phn


def encode_phoneme(sentence: str, phn2id: dict) -> list:
    phonemes = javanese_g2p(sentence)
    return [phn2id[p] for p in phonemes if p in phn2id]


def phonemes_to_text(phonemes: list) -> str:
    result = []
    for p in phonemes:
        if p == '<sp>':   result.append(' ')
        elif p == 'ŋ':    result.append('ng')
        elif p == 'ɲ':    result.append('ny')
        elif p == 'tʰ':   result.append('th')
        elif p == 'dʰ':   result.append('dh')
        else:             result.append(p)
    return ''.join(result).strip()


# ─────────────────────────────────────────────────────────────
#  EKSTRAKSI MFCC (mentah, sebelum normalisasi global)
# ─────────────────────────────────────────────────────────────

try:
    import librosa
    LIBROSA_OK = True
except ImportError:
    LIBROSA_OK = False
try:
    from scipy.io import wavfile
    SCIPY_OK = True
except ImportError:
    SCIPY_OK = False


def extract_features_raw(wav_path):
    """Ekstrak MFCC mentah (tanpa normalisasi per-file)."""
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(wav_path)
    return _librosa_raw(wav_path) if LIBROSA_OK else _manual_raw(wav_path)


def _librosa_raw(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    if len(y) == 0: raise ValueError("Audio kosong")
    y    = np.append(y[0], y[1:] - PRE_EMPH * y[:-1])
    mfcc = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT,
        hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, window="hamming"
    )
    d1   = librosa.feature.delta(mfcc, width=DELTA_WIDTH*2+1)
    d2   = librosa.feature.delta(mfcc, order=2, width=DELTA_WIDTH*2+1)
    feat = np.vstack([mfcc, d1, d2]).T.astype(np.float32)
    # [FIX 2] TIDAK normalisasi per-file di sini
    return feat


def _delta_vec(feat, width=2):
    T=feat.shape[0]; denom=2.0*sum(n*n for n in range(1,width+1))
    delta=np.zeros_like(feat)
    for n in range(1,width+1):
        delta += n*(feat[np.clip(np.arange(T)+n,0,T-1)] -
                    feat[np.clip(np.arange(T)-n,0,T-1)])
    return delta/denom


def _manual_raw(wav_path):
    if not SCIPY_OK: raise ImportError("Butuh librosa atau scipy")
    sr, audio = wavfile.read(wav_path)
    if audio.dtype==np.int16: audio=audio/32768.0
    elif audio.dtype==np.int32: audio=audio/2147483648.0
    audio=audio.astype(np.float32)
    if sr!=SR:
        n=int(len(audio)*SR/sr)
        audio=np.interp(np.linspace(0,len(audio)-1,n),np.arange(len(audio)),audio)
    y=np.append(audio[0],audio[1:]-PRE_EMPH*audio[:-1])
    T_fr=1+(len(y)-WIN_LENGTH)//HOP_LENGTH
    idx=np.arange(WIN_LENGTH)[None,:]+np.arange(T_fr)[:,None]*HOP_LENGTH
    frames=y[idx]*np.hamming(WIN_LENGTH)
    fft_m=np.abs(np.fft.rfft(frames,n=N_FFT))
    def hz2mel(f): return 1127*np.log1p(f/700)
    def mel2hz(m): return 700*(np.exp(m/1127)-1)
    pts=mel2hz(np.linspace(hz2mel(FMIN),hz2mel(FMAX),N_MELS+2))
    bins=np.floor((N_FFT+1)*pts/SR).astype(int)
    fb=np.zeros((N_MELS,N_FFT//2+1))
    for m in range(1,N_MELS+1):
        for k in range(bins[m-1],bins[m]):
            fb[m-1,k]=(k-bins[m-1])/(bins[m]-bins[m-1]+1e-10)
        for k in range(bins[m],bins[m+1]):
            fb[m-1,k]=(bins[m+1]-k)/(bins[m+1]-bins[m]+1e-10)
    log_mel=np.log10(np.maximum(fft_m**2@fb.T,1e-10))
    N_=N_MELS; ni=np.arange(N_MFCC)[:,None]; ki=np.arange(N_)[None,:]+0.5
    mfcc=log_mel@(np.cos(np.pi*ni*ki/N_)*np.sqrt(2/N_)).T
    d1=_delta_vec(mfcc,DELTA_WIDTH); d2=_delta_vec(d1,DELTA_WIDTH)
    feat=np.concatenate([mfcc,d1,d2],axis=1).astype(np.float32)
    return feat


# ─────────────────────────────────────────────────────────────
#  [FIX 2] NORMALISASI MFCC GLOBAL
# ─────────────────────────────────────────────────────────────

def compute_global_norm(sequences: dict, train_ids: list):
    """
    [FIX 2] Hitung mean dan std dari seluruh frame training set.
    Lebih stabil daripada normalisasi per-file.
    """
    print("  Menghitung normalisasi MFCC global dari train set ...")
    all_frames = np.concatenate([sequences[aid] for aid in train_ids], axis=0)
    mean = all_frames.mean(axis=0).astype(np.float32)
    std  = all_frames.std(axis=0).astype(np.float32) + 1e-8
    print(f"  Global norm: mean range [{mean.min():.2f}, {mean.max():.2f}], "
          f"std range [{std.min():.4f}, {std.max():.2f}]")
    return mean, std


def apply_global_norm(sequences: dict, mean: np.ndarray, std: np.ndarray) -> dict:
    """Terapkan normalisasi global ke semua split."""
    return {aid: (feat - mean) / std for aid, feat in sequences.items()}


# ─────────────────────────────────────────────────────────────
#  DATA LOADER
# ─────────────────────────────────────────────────────────────

def find_wav(audio_id):
    for root,_,files in os.walk(SAMPLED_ROOT):
        for f in files:
            if os.path.splitext(f)[0]==audio_id and f.lower().endswith(".wav"):
                return os.path.join(root,f)
    return ""


def load_data(csv_path):
    print(f"  Membaca {csv_path} ...")
    df = pd.read_csv(csv_path)
    print(f"  Shape CSV : {df.shape}")
    print(f"  Kolom     : {list(df.columns)}")

    for col in [COL_PATH_DATA, COL_SENTENCES]:
        if col not in df.columns:
            raise ValueError(f"Kolom '{col}' tidak ada.")

    # [FIX 1] Normalisasi teks sebelum build vocab
    sentences = df[COL_SENTENCES].astype(str).tolist()
    sentences_norm = [normalize_text(s) for s in sentences]
    vocab, phn2id, id2phn = build_phoneme_vocab(sentences_norm)
    print(f"  Vocab fonem : {len(vocab)-1}  (+1 blank → total {len(vocab)})")
    print(f"  Fonem       : {sorted(set(vocab)-{'<blank>'})}")

    sequences_raw, targets, word_refs = {}, {}, {}
    skipped = 0

    print(f"  Mengekstrak MFCC dari {len(df)} file .wav ...")
    for idx, row in df.iterrows():
        audio_id = os.path.splitext(os.path.basename(str(row[COL_PATH_DATA])))[0]
        kalimat  = normalize_text(str(row[COL_SENTENCES]))   # [FIX 1]
        words    = kalimat.split()

        wav_path = str(row[COL_PATH_DATA])
        if not os.path.isfile(wav_path):
            wav_path = find_wav(audio_id)
        if not wav_path:
            skipped += 1; continue

        try:
            feats  = extract_features_raw(wav_path)   # [FIX 2] raw dulu
            T      = feats.shape[0]
            target = encode_phoneme(kalimat, phn2id)

            if T == 0 or not target: skipped+=1; continue
            if T < 2*len(target)+1: skipped+=1; continue

            sequences_raw[audio_id] = feats
            targets[audio_id]       = target
            word_refs[audio_id]     = words

        except Exception as e:
            print(f"  ⚠ {audio_id}: {e}")
            skipped += 1; continue

        if (idx+1)%200==0 or (idx+1)==len(df):
            print(f"    [{idx+1}/{len(df)}] diproses, {skipped} dilewati")

    total_frames = sum(v.shape[0] for v in sequences_raw.values())
    actual_dim   = next(iter(sequences_raw.values())).shape[1]

    print(f"\n  Audio berhasil : {len(sequences_raw)}")
    print(f"  Audio dilewati : {skipped}")
    print(f"  Total frame    : {total_frames:,}")
    print(f"  Input dim      : {actual_dim}")
    print(f"  Output dim     : {len(vocab)}")

    lens = [len(targets[a]) for a in sequences_raw]
    print(f"  Panjang fonem  : min={min(lens)}, mean={np.mean(lens):.1f}, max={max(lens)}")

    return sequences_raw, targets, word_refs, phn2id, id2phn, vocab, actual_dim


# ─────────────────────────────────────────────────────────────
#  CTC LOSS + GRADIENT
# ─────────────────────────────────────────────────────────────

def _softmax(logits):
    x = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)


def ctc_loss_and_grad(logits: np.ndarray, targets: list) -> tuple:
    T, V = logits.shape
    L    = len(targets)
    S    = 2*L + 1
    NEG  = -1e9

    l_ = np.zeros(S, dtype=np.int32)
    for i, c in enumerate(targets):
        l_[2*i+1] = c

    probs     = _softmax(logits).astype(np.float64)
    log_probs = np.log(probs + 1e-30)
    lp_ext    = log_probs[:, l_]

    can_skip = np.zeros(S, dtype=bool)
    for s in range(2, S):
        if l_[s] != BLANK and l_[s] != l_[s-2]:
            can_skip[s] = True
    can_skip_bk = np.zeros(S, dtype=bool)
    for s in range(S-2):
        if l_[s+2] != BLANK and l_[s+2] != l_[s]:
            can_skip_bk[s] = True

    alpha = np.full((T, S), NEG)
    alpha[0, 0] = lp_ext[0, 0]
    if S > 1: alpha[0, 1] = lp_ext[0, 1]

    for t in range(1, T):
        src0 = alpha[t-1]
        src1 = np.empty(S); src1[0]=NEG; src1[1:]=alpha[t-1,:-1]
        src2 = np.empty(S); src2[:2]=NEG; src2[2:]=alpha[t-1,:-2]
        base    = np.logaddexp(src0, src1)
        merged  = np.where(can_skip, np.logaddexp(base, src2), base)
        alpha[t] = merged + lp_ext[t]

    beta = np.full((T, S), NEG)
    beta[T-1, S-1] = 0.0
    if S > 1: beta[T-1, S-2] = 0.0

    for t in range(T-2, -1, -1):
        lp_next = lp_ext[t+1]
        src0 = beta[t+1] + lp_next
        src1 = np.empty(S); src1[-1]=NEG; src1[:-1]=beta[t+1,1:]+lp_next[1:]
        src2 = np.empty(S); src2[-2:]=NEG; src2[:-2]=beta[t+1,2:]+lp_next[2:]
        base    = np.logaddexp(src0, src1)
        beta[t] = np.where(can_skip_bk, np.logaddexp(base, src2), base)

    ll_terms = [alpha[T-1, S-1]]
    if S > 1: ll_terms.append(alpha[T-1, S-2])
    ll = float(logsumexp(ll_terms))

    if not np.isfinite(ll) or ll < -1e7:
        return 0.0, np.zeros((T, V), dtype=np.float32)

    loss = float(-ll / T)

    ab        = alpha + beta
    log_numer = np.full((T, V), NEG)
    for s in range(S):
        log_numer[:, l_[s]] = np.logaddexp(log_numer[:, l_[s]], ab[:, s])

    term = np.exp(np.clip(log_numer - ll, -500, 0))
    return loss, (probs - term).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  [FIX 3] TDNN LAYER dengan Batch Normalization
# ─────────────────────────────────────────────────────────────

class TDNNLayer:
    """
    [FIX 3] Tambah Batch Normalization setelah linear transform.
    BN menstabilkan distribusi aktivasi antar layer → gradien lebih sehat.
    """
    def __init__(self, in_dim, out_dim, context, lr):
        self.context = context
        self.ctx_len = context[1] - context[0] + 1
        self.lr      = lr
        fan_in       = in_dim * self.ctx_len
        self.W = (np.random.randn(fan_in, out_dim) *
                  np.sqrt(2.0/fan_in)).astype(np.float32)
        self.b = np.zeros(out_dim, dtype=np.float32)

        # [FIX 3] BatchNorm parameter
        self.bn_gamma = np.ones(out_dim,  dtype=np.float32)
        self.bn_beta  = np.zeros(out_dim, dtype=np.float32)
        self.bn_eps   = 1e-5
        self.bn_mom   = 0.1    # momentum untuk running stats
        self.bn_run_mean = np.zeros(out_dim, dtype=np.float32)
        self.bn_run_var  = np.ones(out_dim,  dtype=np.float32)

        # Adam untuk W, b, gamma, beta
        self.mW=np.zeros_like(self.W); self.vW=np.zeros_like(self.W)
        self.mb=np.zeros_like(self.b); self.vb=np.zeros_like(self.b)
        self.mg=np.zeros_like(self.bn_gamma); self.vg=np.zeros_like(self.bn_gamma)
        self.mbt=np.zeros_like(self.bn_beta); self.vbt=np.zeros_like(self.bn_beta)
        self.t=0

        self._X=None; self._bn_xhat=None; self._bn_var=None; self._bn_mean=None
        self._dW_acc=np.zeros_like(self.W)
        self._db_acc=np.zeros_like(self.b)
        self._dg_acc=np.zeros_like(self.bn_gamma)
        self._dbt_acc=np.zeros_like(self.bn_beta)
        self._n_acc=0
        self.training = True

    def splice(self, X):
        T=X.shape[0]; idx=np.arange(T)
        return np.concatenate(
            [X[np.clip(idx+off,0,T-1)]
             for off in range(self.context[0],self.context[1]+1)],axis=1)

    def _bn_forward(self, z):
        """[FIX 3] Batch Normalization forward."""
        if self.training:
            mean = z.mean(axis=0)
            var  = z.var(axis=0)
            self.bn_run_mean = (1-self.bn_mom)*self.bn_run_mean + self.bn_mom*mean
            self.bn_run_var  = (1-self.bn_mom)*self.bn_run_var  + self.bn_mom*var
            self._bn_mean = mean
            self._bn_var  = var
        else:
            mean = self.bn_run_mean
            var  = self.bn_run_var

        xhat = (z - mean) / np.sqrt(var + self.bn_eps)
        self._bn_xhat = xhat
        return self.bn_gamma * xhat + self.bn_beta

    def _bn_backward(self, dout):
        """[FIX 3] Batch Normalization backward."""
        T   = dout.shape[0]
        dg  = (dout * self._bn_xhat).sum(axis=0)
        dbt = dout.sum(axis=0)

        dxhat = dout * self.bn_gamma
        var   = self._bn_var + self.bn_eps
        std   = np.sqrt(var)
        dx    = (1.0/T) * (1.0/std) * (
            T*dxhat
            - dxhat.sum(axis=0)
            - self._bn_xhat * (dxhat * self._bn_xhat).sum(axis=0)
        )
        return dx, dg, dbt

    def forward(self, Xsp):
        self._X = Xsp
        z = Xsp @ self.W + self.b
        return self._bn_forward(z)   # [FIX 3]

    def backward(self, dout, accumulate=True):
        T = len(dout)
        # [FIX 3] BN backward dulu
        dz, dg, dbt = self._bn_backward(dout)

        dW  = self._X.T @ dz / T
        db  = dz.mean(0)
        dXsp = dz @ self.W.T

        if accumulate:
            self._dW_acc += dW; self._db_acc  += db
            self._dg_acc += dg; self._dbt_acc += dbt
            self._n_acc  += 1
        else:
            self._adam(dW, db, dg, dbt)

        B, D_tot = dXsp.shape; D = D_tot // self.ctx_len
        dX = np.zeros((B,D), dtype=np.float32)
        for k in range(self.ctx_len):
            dX += dXsp[:, k*D:(k+1)*D]
        return dX

    def flush(self):
        if self._n_acc == 0: return
        n = self._n_acc
        self._adam(self._dW_acc/n, self._db_acc/n,
                   self._dg_acc/n, self._dbt_acc/n)
        self._dW_acc[:]=0; self._db_acc[:]=0
        self._dg_acc[:]=0; self._dbt_acc[:]=0
        self._n_acc = 0

    def set_lr(self, lr):
        self.lr = lr

    def _adam(self, dW, db, dg, dbt, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        # W
        self.mW=b1*self.mW+(1-b1)*dW; self.vW=b2*self.vW+(1-b2)*dW**2
        self.W -= self.lr*(self.mW/(1-b1**self.t))/(np.sqrt(self.vW/(1-b2**self.t))+eps)
        # b
        self.mb=b1*self.mb+(1-b1)*db; self.vb=b2*self.vb+(1-b2)*db**2
        self.b -= self.lr*(self.mb/(1-b1**self.t))/(np.sqrt(self.vb/(1-b2**self.t))+eps)
        # gamma [FIX 3]
        self.mg=b1*self.mg+(1-b1)*dg; self.vg=b2*self.vg+(1-b2)*dg**2
        self.bn_gamma -= self.lr*(self.mg/(1-b1**self.t))/(np.sqrt(self.vg/(1-b2**self.t))+eps)
        # beta [FIX 3]
        self.mbt=b1*self.mbt+(1-b1)*dbt; self.vbt=b2*self.vbt+(1-b2)*dbt**2
        self.bn_beta -= self.lr*(self.mbt/(1-b1**self.t))/(np.sqrt(self.vbt/(1-b2**self.t))+eps)


# ─────────────────────────────────────────────────────────────
#  MODEL TDNN
# ─────────────────────────────────────────────────────────────

class TDNNModel:
    def __init__(self, input_dim, hidden_dims, contexts,
                 output_dim, dropout=0.2, lr=1e-3):
        assert len(hidden_dims)==len(contexts)
        self.dropout=dropout; self.training=True
        self.layers=[]
        in_d=input_dim
        for h,ctx in zip(hidden_dims,contexts):
            self.layers.append(TDNNLayer(in_d,h,ctx,lr)); in_d=h
        # Layer output tidak pakai BN
        self.layers.append(TDNNLayer(in_d,output_dim,[0,0],lr))
        self._rc=[]; self._dc=[]

    def set_training(self, mode: bool):
        """[FIX 4] Pastikan semua layer sinkron dengan mode training."""
        self.training = mode
        for layer in self.layers:
            layer.training = mode

    def set_lr(self, lr):
        for layer in self.layers:
            layer.set_lr(lr)

    def forward(self, X):
        self._rc=[]; self._dc=[]; h=X
        for layer in self.layers[:-1]:
            z=layer.forward(layer.splice(h))
            a=np.maximum(0.0,z)
            if self.training and self.dropout>0:
                mask=(np.random.rand(*a.shape)>self.dropout).astype(np.float32)
                a=a*mask/(1.0-self.dropout)
            else:
                mask=np.ones(a.shape,dtype=np.float32)
            self._rc.append(z); self._dc.append(mask); h=a
        out=self.layers[-1]
        return out.forward(out.splice(h))

    def ctc_step(self, X, target, accumulate=True):
        logits=self.forward(X)
        loss,dlogit=ctc_loss_and_grad(logits,target)
        if loss==0.0: return 0.0
        gnorm=np.sqrt((dlogit**2).sum())
        if gnorm>GRAD_CLIP: dlogit=dlogit*(GRAD_CLIP/gnorm)
        sc=1.0/(1.0-self.dropout+1e-9) if self.dropout>0 else 1.0
        dh=self.layers[-1].backward(dlogit,accumulate=accumulate)
        for i in reversed(range(len(self.layers)-1)):
            dh=dh*self._dc[i]*sc
            dh=dh*(self._rc[i]>0).astype(np.float32)
            dh=self.layers[i].backward(dh,accumulate=accumulate)
        return loss

    def flush(self):
        for layer in self.layers: layer.flush()

    def get_log_probs(self, X):
        self.set_training(False)   # [FIX 4]
        p=_softmax(self.forward(X))
        return np.log(p+1e-30)

    def save(self, path):
        data = {
            "w": [(l.W.copy(), l.b.copy()) for l in self.layers],
            "bn": [(l.bn_gamma.copy(), l.bn_beta.copy(),
                    l.bn_run_mean.copy(), l.bn_run_var.copy())
                   for l in self.layers],
        }
        np.save(path, data, allow_pickle=True)

    def load(self, path):
        d = np.load(path, allow_pickle=True).item()
        for i, (W, b) in enumerate(d["w"]):
            self.layers[i].W = W; self.layers[i].b = b
        if "bn" in d:
            for i, (g, bt, rm, rv) in enumerate(d["bn"]):
                self.layers[i].bn_gamma    = g
                self.layers[i].bn_beta     = bt
                self.layers[i].bn_run_mean = rm
                self.layers[i].bn_run_var  = rv
        print(f"  Model dimuat: {path}")


# ─────────────────────────────────────────────────────────────
#  SPLIT
# ─────────────────────────────────────────────────────────────

def split_dataset(ids):
    rng=np.random.default_rng(RANDOM_SEED)
    ids=list(ids); perm=rng.permutation(len(ids))
    n=len(ids); ntr=int(n*TRAIN_RATIO); nva=int(n*VAL_RATIO)
    tr=[ids[i] for i in perm[:ntr]]
    va=[ids[i] for i in perm[ntr:ntr+nva]]
    te=[ids[i] for i in perm[ntr+nva:]]
    print(f"  Split → train:{len(tr)}  val:{len(va)}  test:{len(te)}")
    return tr,va,te


# ─────────────────────────────────────────────────────────────
#  TRAINING
# ─────────────────────────────────────────────────────────────

def get_lr(epoch: int, no_improve_count: int) -> float:
    if epoch <= WARMUP_EPOCH:
        alpha = (epoch - 1) / max(WARMUP_EPOCH - 1, 1)
        return LR_WARMUP + alpha * (LR_INIT - LR_WARMUP)
    else:
        n_decay = no_improve_count // LR_PATIENCE
        return max(LR_INIT * (LR_DECAY ** n_decay), LR_MIN)


def compute_wer_sample(model, ids, sequences, word_refs, id2phn,
                       n_sample=50, rng=None):
    """[FIX 5] Hitung WER cepat pada sample kecil untuk monitoring."""
    if rng is None: rng = np.random.default_rng(42)
    sample = list(rng.choice(ids, min(n_sample, len(ids)), replace=False))
    model.set_training(False)
    tS=tD=tI=tN=0
    for aid in sample:
        lp = model.get_log_probs(sequences[aid])
        decoded = ctc_beam_decode(lp, id2phn, beam_width=5)  # beam kecil untuk kecepatan
        hw = decoded.split()
        rw = word_refs.get(aid, [])
        _, S, D, I, N = levenshtein(rw, hw)
        tS+=S; tD+=D; tI+=I; tN+=N
    return (tS+tD+tI)/max(tN,1)*100


def train(model, train_ids, val_ids, sequences, targets, word_refs, id2phn):
    rng=np.random.default_rng(RANDOM_SEED)
    hist=[]; best=float("inf"); pat=0; no_improve=0

    print(f"\n  Train: {len(train_ids)} audio | Val: {len(val_ids)} audio")
    print(f"  Epochs: {MAX_EPOCHS} | EarlyStop: {EARLY_STOP} | Batch: {BATCH_SIZE}")
    print(f"  LR: warmup {LR_WARMUP}→{LR_INIT} ({WARMUP_EPOCH} ep), "
          f"decay ×{LR_DECAY} tiap {LR_PATIENCE} ep")
    header = f"  {'Ep':>4}  {'LR':>8}  {'TrLoss':>8}  {'VaLoss':>8}  {'TrWER%':>7}  {'t':>7}"
    print(f"\n{header}")
    print(f"  {'-'*54}")

    for epoch in range(1, MAX_EPOCHS+1):
        t0=time.time()

        lr=get_lr(epoch, no_improve)
        model.set_lr(lr)
        model.set_training(True)   # [FIX 4]

        perm=rng.permutation(len(train_ids))
        order=[train_ids[i] for i in perm]
        total_loss=0.0; n_batches=0

        for start in range(0, len(order), BATCH_SIZE):
            batch=order[start:start+BATCH_SIZE]
            batch_loss=0.0; valid=0
            for aid in batch:
                try:
                    lv=model.ctc_step(sequences[aid],targets[aid],
                                      accumulate=True)
                    if lv>0: batch_loss+=lv; valid+=1
                except Exception: continue
            model.flush()
            if valid>0: total_loss+=batch_loss/valid; n_batches+=1

        tr_loss=total_loss/max(n_batches,1)

        # [FIX 4] Validasi dengan training=False konsisten
        model.set_training(False)
        val_loss=0.0; val_n=0
        for aid in val_ids:
            try:
                logits=model.forward(sequences[aid])
                lv,_=ctc_loss_and_grad(logits,targets[aid])
                if lv>0: val_loss+=lv; val_n+=1
            except Exception: continue
        val_loss/=max(val_n,1)

        elapsed=time.time()-t0

        # [FIX 5] Hitung WER train setiap 5 epoch
        tr_wer_str = "   -  "
        if epoch % 5 == 0 or epoch == 1:
            tr_wer = compute_wer_sample(model, train_ids, sequences,
                                        word_refs, id2phn, n_sample=50, rng=rng)
            tr_wer_str = f"{tr_wer:6.1f}%"

        hist.append({"epoch":epoch,"lr":round(lr,8),
                     "train_loss":round(tr_loss,6),
                     "val_loss":round(val_loss,6)})
        print(f"  {epoch:4d}  {lr:8.6f}  {tr_loss:8.4f}  "
              f"{val_loss:8.4f}  {tr_wer_str}  {elapsed:6.1f}s")

        if val_loss < best-1e-5:
            best=val_loss; pat=0; no_improve=0
            model.save(MODEL_PATH)
        else:
            pat+=1; no_improve+=1
            if pat>=EARLY_STOP:
                print(f"\n  ✓ Early stop epoch {epoch} (best val_loss={best:.4f})")
                break

    return hist


# ─────────────────────────────────────────────────────────────
#  [FIX 6] SIMPLE BEAM SEARCH CTC DECODE
# ─────────────────────────────────────────────────────────────

def ctc_beam_decode(log_probs: np.ndarray, id2phn: dict,
                    beam_width: int = 10) -> str:
    """
    [FIX 6] Beam search CTC decode.
    Lebih baik dari greedy untuk menghindari fonem berulang.

    Beam state: dict { (sequence_tuple): log_prob }
    Dua beam: Pb (diakhiri blank), Pnb (diakhiri non-blank)
    """
    T, V = log_probs.shape
    NEG  = -float('inf')

    # Init: beam kosong
    # beam = { sequence: (log_Pb, log_Pnb) }
    beam = {(): (0.0, NEG)}

    for t in range(T):
        new_beam = {}

        for seq, (pb, pnb) in beam.items():
            # Extend with blank
            lp_blank = log_probs[t, BLANK]
            prev_pb, prev_pnb = new_beam.get(seq, (NEG, NEG))
            new_pb = np.logaddexp(prev_pb,
                                   np.logaddexp(pb, pnb) + lp_blank)
            new_beam[seq] = (new_pb, new_beam.get(seq, (NEG,NEG))[1])

            # Extend with each non-blank
            for c in range(1, V):
                lp_c = log_probs[t, c]
                new_seq = seq + (c,)

                # Jika c sama dengan last char di seq
                if seq and seq[-1] == c:
                    # Hanya bisa extend dari blank path
                    ep_nb = pb + lp_c
                else:
                    ep_nb = np.logaddexp(pb, pnb) + lp_c

                prev_pb2, prev_pnb2 = new_beam.get(new_seq, (NEG, NEG))
                new_beam[new_seq] = (prev_pb2, np.logaddexp(prev_pnb2, ep_nb))

        # Prune ke beam_width terbaik berdasarkan total log prob
        beam = dict(
            sorted(new_beam.items(),
                   key=lambda kv: np.logaddexp(kv[1][0], kv[1][1]),
                   reverse=True)[:beam_width]
        )

    # Ambil sequence terbaik
    best_seq = max(beam.items(),
                   key=lambda kv: np.logaddexp(kv[1][0], kv[1][1]))[0]

    phonemes = [id2phn.get(int(c), '') for c in best_seq if c != BLANK]
    return phonemes_to_text(phonemes)


def ctc_greedy_decode(log_probs: np.ndarray, id2phn: dict) -> str:
    """Greedy decode (fallback / baseline)."""
    preds=np.argmax(log_probs,axis=1)
    phonemes=[]; prev=None
    for p in preds:
        if p!=prev:
            if p!=BLANK:
                phonemes.append(id2phn.get(int(p),""))
            prev=p
    return phonemes_to_text(phonemes)


# ─────────────────────────────────────────────────────────────
#  WER
# ─────────────────────────────────────────────────────────────

def levenshtein(ref, hyp):
    R,H=len(ref),len(hyp)
    dp=np.zeros((R+1,H+1),dtype=int)
    dp[:,0]=np.arange(R+1); dp[0,:]=np.arange(H+1)
    for i in range(1,R+1):
        for j in range(1,H+1):
            c=0 if ref[i-1]==hyp[j-1] else 1
            dp[i,j]=min(dp[i-1,j]+1,dp[i,j-1]+1,dp[i-1,j-1]+c)
    i,j=R,H; S=D=I=0
    while i>0 or j>0:
        if i>0 and j>0:
            c=0 if ref[i-1]==hyp[j-1] else 1
            if dp[i,j]==dp[i-1,j-1]+c:
                if c: S+=1
                i-=1;j-=1
            elif dp[i,j]==dp[i-1,j]+1: D+=1;i-=1
            else: I+=1;j-=1
        elif i>0: D+=1;i-=1
        else: I+=1;j-=1
    return min((S+D+I)/max(R,1),1.0),S,D,I,max(R,1)


# ─────────────────────────────────────────────────────────────
#  EVALUASI
# ─────────────────────────────────────────────────────────────

def evaluate(model, eval_ids, sequences, word_refs, id2phn, split_name="eval"):
    model.set_training(False)   # [FIX 4]
    print(f"\n  Mendekode {len(eval_ids)} audio ({split_name}) ...")
    wer_rows=[]; tS=tD=tI=tN=0

    for aid in eval_ids:
        lp=model.get_log_probs(sequences[aid])
        # [FIX 6] Gunakan beam search
        decoded=ctc_beam_decode(lp, id2phn, beam_width=BEAM_WIDTH)
        hw=decoded.split()
        rw=word_refs.get(aid,[])
        wv,S,D,I,N=levenshtein(rw,hw)
        tS+=S;tD+=D;tI+=I;tN+=N
        wer_rows.append({"audio_id":aid,"WER":round(wv*100,4),
                         "S":S,"D":D,"I":I,"N":N,
                         "referensi":" ".join(rw),
                         "hipotesis":decoded})

    wc=(tS+tD+tI)/max(tN,1)*100
    wl=[r["WER"] for r in wer_rows]
    s={"split":split_name,
       "n_audio":len(eval_ids),
       "wer_corpus":round(wc,4),
       "wer_mean":round(float(np.mean(wl)),4),
       "wer_min":round(float(np.min(wl)),4),
       "wer_max":round(float(np.max(wl)),4),
       "total_S":tS,"total_D":tD,"total_I":tI,"total_N":tN}

    print(f"\n── Statistik Evaluasi TDNN-CTC v5 [{split_name}] ─────────────")
    print(f"  Jumlah audio   : {s['n_audio']}")
    print(f"  WER Corpus     : {s['wer_corpus']:.2f}%")
    print(f"  WER Mean       : {s['wer_mean']:.2f}%")
    print(f"  WER Min        : {s['wer_min']:.2f}%")
    print(f"  WER Max        : {s['wer_max']:.2f}%")
    print(f"  Total S/D/I/N  : {s['total_S']}/{s['total_D']}/{s['total_I']}/{s['total_N']}")
    print(f"────────────────────────────────────────────────────────────")
    return wer_rows, s


# ─────────────────────────────────────────────────────────────
#  PERBANDINGAN
# ─────────────────────────────────────────────────────────────

def print_comparison(s_v4, s_v5):
    g={"wer_corpus":99.84,"wer_mean":99.82}
    print(f"\n{'='*68}")
    print(f"  PERBANDINGAN GMM vs TDNN-CTC v4 vs TDNN-CTC v5 (Phoneme + Fix)")
    print(f"{'='*68}")
    print(f"  {'Metrik':<30} {'GMM':>10} {'v4':>10} {'v5 (fix)':>10}")
    print(f"  {'-'*62}")
    for lbl, gv, v4v, v5v in [
        ("WER Corpus (%)",
         f"{g['wer_corpus']:.2f}",
         f"{s_v4.get('wer_corpus', '-'):.2f}" if isinstance(s_v4.get('wer_corpus'), float) else "-",
         f"{s_v5['wer_corpus']:.2f}"),
        ("WER Mean (%)",
         f"{g['wer_mean']:.2f}",
         f"{s_v4.get('wer_mean', '-'):.2f}" if isinstance(s_v4.get('wer_mean'), float) else "-",
         f"{s_v5['wer_mean']:.2f}"),
    ]:
        print(f"  {lbl:<30} {gv:>10} {v4v:>10} {v5v:>10}")
    print(f"{'='*68}")
    d = g["wer_corpus"] - s_v5["wer_corpus"]
    if d > 0: print(f"  ✓ v5 lebih baik {d:.2f}% dari GMM")
    elif d < 0: print(f"  ✗ v5 lebih buruk {abs(d):.2f}% dari GMM")
    else: print(f"  = WER sama dengan GMM")


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    np.random.seed(RANDOM_SEED)

    print("="*62)
    print("  TAHAP 1 — Memuat Data & Ekstraksi MFCC dari WAV")
    print("="*62)
    print(f"  Folder audio : {SAMPLED_ROOT}")
    print(f"  CSV          : {CSV_PATH}")
    print(f"  Output       : {DIR_SAVE}")
    print(f"  Label        : fonem Bahasa Jawa via G2P + normalisasi [FIX 1]")
    print(f"  MFCC norm    : global dari train set [FIX 2]")

    (sequences_raw, targets, word_refs,
     phn2id, id2phn, vocab, actual_dim) = load_data(CSV_PATH)

    all_ids = list(sequences_raw.keys())
    train_ids, val_ids, test_ids = split_dataset(all_ids)

    # [FIX 2] Normalisasi MFCC global
    mean_g, std_g = compute_global_norm(sequences_raw, train_ids)
    np.save(NORM_PATH, {"mean": mean_g, "std": std_g})
    sequences = apply_global_norm(sequences_raw, mean_g, std_g)
    print(f"  Normalisasi global tersimpan: {NORM_PATH}")

    print("\n"+"="*62)
    print("  TAHAP 2 — Bangun & Latih Model TDNN + CTC v5")
    print("="*62)
    print(f"  Input dim    : {actual_dim}")
    for i,(h,ctx) in enumerate(zip(HIDDEN_DIMS,TDNN_CONTEXTS)):
        print(f"  Layer {i+1}       : {h} unit  context {ctx}  + BatchNorm [FIX 3]")
    print(f"  Output dim   : {len(vocab)}  (fonem + blank)")
    print(f"  Loss         : CTC phoneme-level")
    print(f"  Dropout      : {DROPOUT_RATE}")
    print(f"  Decode       : Beam Search width={BEAM_WIDTH} [FIX 6]")

    model = TDNNModel(
        input_dim=actual_dim, hidden_dims=HIDDEN_DIMS,
        contexts=TDNN_CONTEXTS, output_dim=len(vocab),
        dropout=DROPOUT_RATE, lr=LR_WARMUP,
    )

    history = train(model, train_ids, val_ids, sequences,
                    targets, word_refs, id2phn)
    pd.DataFrame(history).to_csv(OUT_HISTORY_CSV, index=False)
    print(f"  History tersimpan: {OUT_HISTORY_CSV}")

    if os.path.exists(MODEL_PATH):
        model.load(MODEL_PATH)

    print("\n"+"="*62)
    print("  TAHAP 3 — Evaluasi WER (Val + Test)")
    print("="*62)

    # [FIX 5] Evaluasi terpisah: val dan test
    wer_rows_val,  summ_val  = evaluate(model, val_ids,  sequences, word_refs, id2phn, "val")
    wer_rows_test, summ_test = evaluate(model, test_ids, sequences, word_refs, id2phn, "test")

    all_rows = wer_rows_val + wer_rows_test
    df_w = pd.DataFrame(all_rows)
    df_w[["audio_id","WER","S","D","I","N"]].to_csv(OUT_WER_CSV, index=False)
    df_w[["audio_id","referensi","hipotesis","WER"]].to_csv(OUT_DEC_CSV, index=False)
    print(f"\n  Tersimpan:\n  {OUT_WER_CSV}\n  {OUT_DEC_CSV}")

    print("\n"+"="*62)
    print("  TAHAP 4 — Perbandingan")
    print("="*62)
    # v4 baseline (dari run sebelumnya jika ada)
    v4_dummy = {"wer_corpus": None, "wer_mean": None}
    print_comparison(v4_dummy, summ_test)

    return model, summ_test, history


if __name__ == "__main__":
    model, summary, history = main()

In [ ]:
"""
============================================================
  TDNN + CTC v5b (Revised) — Javanese ASR (Phoneme-level)
  PERBAIKAN:
  1. Arsitektur disesuaikan tepat 5 Layer (1 Input + 4 Hidden) + 1 Output Layer.
  2. Pola konteks disesuaikan dengan teori:
     [-2,2], [-1,2], [-3,3], [-7,2], [0,0].
  3. Pembersihan logika iterasi pada pembuatan layer di model.
============================================================
"""

import os, time, warnings
import numpy as np
import pandas as pd
from scipy.special import logsumexp
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  KONFIGURASI
# ─────────────────────────────────────────────────────────────

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/GMM_TDNN/out/test_5"
)
os.makedirs(DIR_SAVE, exist_ok=True)

OUT_WER_CSV     = os.path.join(DIR_SAVE, "HasilWER_TDNN_v5b_revised.csv")
OUT_DEC_CSV     = os.path.join(DIR_SAVE, "HasilDekode_TDNN_v5b_revised.csv")
OUT_HISTORY_CSV = os.path.join(DIR_SAVE, "HasilTraining_TDNN_v5b_revised.csv")
MODEL_PATH      = os.path.join(DIR_SAVE, "tdnn_v5b_revised_best.npy")
NORM_PATH       = os.path.join(DIR_SAVE, "mfcc_global_norm.npy")

COL_PATH_DATA = "Path_Data"
COL_SENTENCES = "sentences"

SR=16000; N_MFCC=13; N_FFT=512; HOP_LENGTH=160
WIN_LENGTH=400; N_MELS=26; PRE_EMPH=0.97
FMIN=0.0; FMAX=4000.0; DELTA_WIDTH=2
INPUT_DIM = N_MFCC * 3   # 39

# [PERBAIKAN] Arsitektur disesuaikan dengan instruksi (5 Layer)
# Layer 1: Input Layer [-2, 2]
# Layer 2: Hidden 1 [-1, 2]
# Layer 3: Hidden 2 [-3, 3]
# Layer 4: Hidden 3 [-7, 2]
# Layer 5: Hidden 4 [0, 0]
TDNN_CONTEXTS = [[-2, 2], [-1, 2], [-3, 3], [-7, 2], [0, 0]]
HIDDEN_DIMS   = [256, 256, 256, 256, 256]

DROPOUT_RATE  = 0.2
LR_INIT       = 1e-3
LR_WARMUP     = 1e-5
GRAD_CLIP     = 1.0

BATCH_SIZE   = 16
MAX_EPOCHS   = 60
EARLY_STOP   = 12
WARMUP_EPOCH = 5
LR_DECAY     = 0.5
LR_PATIENCE  = 5
LR_MIN       = 1e-6

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
RANDOM_SEED = 42
BLANK       = 0
BEAM_WIDTH  = 10


# ─────────────────────────────────────────────────────────────
#  [FIX 1] NORMALISASI TEKS + G2P
# ─────────────────────────────────────────────────────────────

_ANGKA_JAWA = {
    0:'nol',1:'siji',2:'loro',3:'telu',4:'papat',5:'lima',
    6:'enem',7:'pitu',8:'wolu',9:'songo',10:'sepuluh',
    11:'sewelas',12:'rolas',13:'telulas',14:'patbelas',
    15:'limolas',16:'nembelas',17:'pitulas',18:'wolulas',
    19:'sangalas',20:'rongpuluh',
}

def _angka_ke_jawa(n: int) -> str:
    if n in _ANGKA_JAWA: return _ANGKA_JAWA[n]
    if n < 100:
        p = _ANGKA_JAWA.get(n//10, str(n//10)) + "puluh"
        s = n % 10
        return p if s==0 else p + _ANGKA_JAWA.get(s, str(s))
    if n < 1000:
        r = n // 100; sisa = n % 100
        pfx = "satus" if r==1 else _ANGKA_JAWA.get(r, str(r)) + "atus"
        return pfx if sisa==0 else pfx + _angka_ke_jawa(sisa)
    r = n // 1000; sisa = n % 1000
    pfx = "sewu" if r==1 else _ANGKA_JAWA.get(r, str(r)) + "ewu"
    return pfx if sisa==0 else pfx + _angka_ke_jawa(sisa)

def normalize_text(text: str) -> str:
    import re
    text = text.lower().strip()
    text = re.sub(r'\d+', lambda m: ' ' + _angka_ke_jawa(int(m.group())) + ' ', text)
    text = re.sub(r"[^a-zéèêâ\s]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

DIGRAPHS = {'ng': 'ŋ', 'ny': 'ɲ', 'th': 'tʰ', 'dh': 'dʰ'}

def javanese_g2p(text: str) -> list:
    text = normalize_text(text)
    phonemes = []; i = 0
    while i < len(text):
        ch = text[i]
        if ch == ' ':
            if phonemes and phonemes[-1] != '<sp>': phonemes.append('<sp>')
            i += 1; continue
        if not ch.isalpha() and ch not in 'éèâê': i += 1; continue
        if i+1 < len(text):
            two = text[i:i+2]
            if two in DIGRAPHS: phonemes.append(DIGRAPHS[two]); i+=2; continue
        phonemes.append(ch); i += 1
    return phonemes

def build_phoneme_vocab(sentences):
    all_phn = set()
    for s in sentences: all_phn.update(javanese_g2p(s))
    vocab  = ['<blank>'] + sorted(all_phn)
    phn2id = {p: i for i,p in enumerate(vocab)}
    id2phn = {i: p for p,i in phn2id.items()}
    return vocab, phn2id, id2phn

def encode_phoneme(sentence, phn2id):
    return [phn2id[p] for p in javanese_g2p(sentence) if p in phn2id]

def phonemes_to_text(phonemes):
    r = []
    for p in phonemes:
        if p=='<sp>': r.append(' ')
        elif p=='ŋ':  r.append('ng')
        elif p=='ɲ':  r.append('ny')
        elif p=='tʰ': r.append('th')
        elif p=='dʰ': r.append('dh')
        else:         r.append(p)
    return ''.join(r).strip()


# ─────────────────────────────────────────────────────────────
#  EKSTRAKSI MFCC
# ─────────────────────────────────────────────────────────────

try:
    import librosa; LIBROSA_OK = True
except ImportError:
    LIBROSA_OK = False
try:
    from scipy.io import wavfile; SCIPY_OK = True
except ImportError:
    SCIPY_OK = False

def extract_features_raw(wav_path):
    if not os.path.isfile(wav_path): raise FileNotFoundError(wav_path)
    return _librosa_raw(wav_path) if LIBROSA_OK else _manual_raw(wav_path)

def _librosa_raw(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    if len(y)==0: raise ValueError("Audio kosong")
    y    = np.append(y[0], y[1:] - PRE_EMPH*y[:-1])
    mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT,
               hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
               n_mels=N_MELS, fmin=FMIN, fmax=FMAX, window="hamming")
    d1 = librosa.feature.delta(mfcc, width=DELTA_WIDTH*2+1)
    d2 = librosa.feature.delta(mfcc, order=2, width=DELTA_WIDTH*2+1)
    return np.vstack([mfcc,d1,d2]).T.astype(np.float32)

def _delta_vec(feat, width=2):
    T=feat.shape[0]; denom=2.0*sum(n*n for n in range(1,width+1))
    delta=np.zeros_like(feat)
    for n in range(1,width+1):
        delta += n*(feat[np.clip(np.arange(T)+n,0,T-1)] -
                    feat[np.clip(np.arange(T)-n,0,T-1)])
    return delta/denom

def _manual_raw(wav_path):
    if not SCIPY_OK: raise ImportError("Butuh librosa atau scipy")
    sr, audio = wavfile.read(wav_path)
    if audio.dtype==np.int16: audio=audio/32768.0
    elif audio.dtype==np.int32: audio=audio/2147483648.0
    audio=audio.astype(np.float32)
    if sr!=SR:
        n=int(len(audio)*SR/sr)
        audio=np.interp(np.linspace(0,len(audio)-1,n),np.arange(len(audio)),audio)
    y=np.append(audio[0],audio[1:]-PRE_EMPH*audio[:-1])
    T_fr=1+(len(y)-WIN_LENGTH)//HOP_LENGTH
    idx=np.arange(WIN_LENGTH)[None,:]+np.arange(T_fr)[:,None]*HOP_LENGTH
    frames=y[idx]*np.hamming(WIN_LENGTH)
    fft_m=np.abs(np.fft.rfft(frames,n=N_FFT))
    def hz2mel(f): return 1127*np.log1p(f/700)
    def mel2hz(m): return 700*(np.exp(m/1127)-1)
    pts=mel2hz(np.linspace(hz2mel(FMIN),hz2mel(FMAX),N_MELS+2))
    bins=np.floor((N_FFT+1)*pts/SR).astype(int)
    fb=np.zeros((N_MELS,N_FFT//2+1))
    for m in range(1,N_MELS+1):
        for k in range(bins[m-1],bins[m]):
            fb[m-1,k]=(k-bins[m-1])/(bins[m]-bins[m-1]+1e-10)
        for k in range(bins[m],bins[m+1]):
            fb[m-1,k]=(bins[m+1]-k)/(bins[m+1]-bins[m]+1e-10)
    log_mel=np.log10(np.maximum(fft_m**2@fb.T,1e-10))
    N_=N_MELS; ni=np.arange(N_MFCC)[:,None]; ki=np.arange(N_)[None,:]+0.5
    mfcc=log_mel@(np.cos(np.pi*ni*ki/N_)*np.sqrt(2/N_)).T
    d1=_delta_vec(mfcc,DELTA_WIDTH); d2=_delta_vec(d1,DELTA_WIDTH)
    return np.concatenate([mfcc,d1,d2],axis=1).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  [FIX 2] NORMALISASI MFCC GLOBAL
# ─────────────────────────────────────────────────────────────

def compute_global_norm(sequences, train_ids):
    print("  Menghitung normalisasi MFCC global dari train set ...")
    all_frames = np.concatenate([sequences[aid] for aid in train_ids], axis=0)
    mean = all_frames.mean(axis=0).astype(np.float32)
    std  = all_frames.std(axis=0).astype(np.float32) + 1e-8
    print(f"  mean [{mean.min():.2f}, {mean.max():.2f}]  "
          f"std [{std.min():.4f}, {std.max():.2f}]")
    return mean, std

def apply_global_norm(sequences, mean, std):
    return {aid: (feat - mean) / std for aid, feat in sequences.items()}


# ─────────────────────────────────────────────────────────────
#  DATA LOADER
# ─────────────────────────────────────────────────────────────

def find_wav(audio_id):
    for root,_,files in os.walk(SAMPLED_ROOT):
        for f in files:
            if os.path.splitext(f)[0]==audio_id and f.lower().endswith(".wav"):
                return os.path.join(root,f)
    return ""

def load_data(csv_path):
    print(f"  Membaca {csv_path} ...")
    df = pd.read_csv(csv_path)
    for col in [COL_PATH_DATA, COL_SENTENCES]:
        if col not in df.columns: raise ValueError(f"Kolom '{col}' tidak ada.")

    sentences_norm = [normalize_text(str(s)) for s in df[COL_SENTENCES].tolist()]
    vocab, phn2id, id2phn = build_phoneme_vocab(sentences_norm)
    print(f"  Vocab fonem : {len(vocab)-1}  (+1 blank → total {len(vocab)})")

    sequences_raw, targets, word_refs = {}, {}, {}
    skipped = 0

    print(f"  Mengekstrak MFCC dari {len(df)} file .wav ...")
    for idx, row in df.iterrows():
        audio_id = os.path.splitext(os.path.basename(str(row[COL_PATH_DATA])))[0]
        kalimat  = normalize_text(str(row[COL_SENTENCES]))
        words    = kalimat.split()
        wav_path = str(row[COL_PATH_DATA])
        if not os.path.isfile(wav_path): wav_path = find_wav(audio_id)
        if not wav_path: skipped+=1; continue
        try:
            feats  = extract_features_raw(wav_path)
            T      = feats.shape[0]
            target = encode_phoneme(kalimat, phn2id)
            if T==0 or not target: skipped+=1; continue
            if T < 2*len(target)+1: skipped+=1; continue
            sequences_raw[audio_id] = feats
            targets[audio_id]       = target
            word_refs[audio_id]     = words
        except Exception as e:
            skipped+=1; continue

    actual_dim = next(iter(sequences_raw.values())).shape[1]
    print(f"\n  Audio berhasil : {len(sequences_raw)} | dilewati : {skipped}")
    return sequences_raw, targets, word_refs, phn2id, id2phn, vocab, actual_dim


# ─────────────────────────────────────────────────────────────
#  CTC LOSS + GRADIENT
# ─────────────────────────────────────────────────────────────

def _softmax(logits):
    x = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def ctc_loss_and_grad(logits, targets):
    T, V = logits.shape; L = len(targets); S = 2*L+1; NEG = -1e9
    l_ = np.zeros(S, dtype=np.int32)
    for i,c in enumerate(targets): l_[2*i+1] = c
    probs     = _softmax(logits).astype(np.float64)
    log_probs = np.log(probs+1e-30); lp_ext = log_probs[:,l_]
    can_skip = np.zeros(S,dtype=bool)
    for s in range(2,S):
        if l_[s]!=BLANK and l_[s]!=l_[s-2]: can_skip[s]=True
    can_skip_bk = np.zeros(S,dtype=bool)
    for s in range(S-2):
        if l_[s+2]!=BLANK and l_[s+2]!=l_[s]: can_skip_bk[s]=True
    
    alpha = np.full((T,S),NEG); alpha[0,0]=lp_ext[0,0]
    if S>1: alpha[0,1]=lp_ext[0,1]
    for t in range(1,T):
        s0=alpha[t-1]; s1=np.empty(S); s1[0]=NEG; s1[1:]=alpha[t-1,:-1]
        s2=np.empty(S); s2[:2]=NEG; s2[2:]=alpha[t-1,:-2]
        alpha[t]=np.where(can_skip,np.logaddexp(np.logaddexp(s0,s1),s2),
                          np.logaddexp(s0,s1))+lp_ext[t]
                          
    beta = np.full((T,S),NEG); beta[T-1,S-1]=0.0
    if S>1: beta[T-1,S-2]=0.0
    for t in range(T-2,-1,-1):
        lp=lp_ext[t+1]; s0=beta[t+1]+lp
        s1=np.empty(S); s1[-1]=NEG; s1[:-1]=beta[t+1,1:]+lp[1:]
        s2=np.empty(S); s2[-2:]=NEG; s2[:-2]=beta[t+1,2:]+lp[2:]
        beta[t]=np.where(can_skip_bk,np.logaddexp(np.logaddexp(s0,s1),s2),
                         np.logaddexp(s0,s1))
                         
    ll_terms=[alpha[T-1,S-1]]
    if S>1: ll_terms.append(alpha[T-1,S-2])
    ll = float(logsumexp(ll_terms))
    if not np.isfinite(ll) or ll < -1e7:
        return 0.0, np.zeros((T,V),dtype=np.float32)
        
    loss = float(-ll/T)
    ab=alpha+beta; log_numer=np.full((T,V),NEG)
    for s in range(S):
        log_numer[:,l_[s]]=np.logaddexp(log_numer[:,l_[s]],ab[:,s])
    term = np.exp(np.clip(log_numer-ll,-500,0))
    return loss, (probs-term).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  LAYER NORM
# ─────────────────────────────────────────────────────────────

class LayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.eps   = eps
        self.gamma = np.ones(dim,  dtype=np.float32)
        self.beta  = np.zeros(dim, dtype=np.float32)
        self.mg=np.zeros(dim,dtype=np.float32); self.vg=np.zeros(dim,dtype=np.float32)
        self.mb=np.zeros(dim,dtype=np.float32); self.vb=np.zeros(dim,dtype=np.float32)
        self.t=0
        self._xhat=None; self._std=None
        self._dg_acc=np.zeros(dim,dtype=np.float32)
        self._db_acc=np.zeros(dim,dtype=np.float32)
        self._n_acc=0

    def forward(self, x):
        mean = x.mean(axis=1, keepdims=True)
        var  = x.var(axis=1,  keepdims=True)
        std  = np.sqrt(var + self.eps)
        xhat = (x - mean) / std
        self._xhat = xhat; self._std = std
        return self.gamma * xhat + self.beta

    def backward(self, dout, accumulate=True):
        T, D = dout.shape
        dg  = (dout * self._xhat).sum(axis=0)
        db  = dout.sum(axis=0)
        dxhat = dout * self.gamma
        dx = (1.0/D) * (1.0/self._std) * (
            D*dxhat
            - dxhat.sum(axis=1, keepdims=True)
            - self._xhat * (dxhat * self._xhat).sum(axis=1, keepdims=True)
        )
        if accumulate:
            self._dg_acc += dg; self._db_acc += db; self._n_acc += 1
        else:
            self._adam(dg, db)
        return dx

    def flush(self):
        if self._n_acc==0: return
        self._adam(self._dg_acc/self._n_acc, self._db_acc/self._n_acc)
        self._dg_acc[:]=0; self._db_acc[:]=0; self._n_acc=0

    def set_lr(self, lr): self.lr = lr

    def _adam(self, dg, db, lr=None, b1=0.9, b2=0.999, eps=1e-8):
        if lr is None: lr = getattr(self, 'lr', 1e-3)
        self.t += 1
        self.mg=b1*self.mg+(1-b1)*dg; self.vg=b2*self.vg+(1-b2)*dg**2
        self.gamma -= lr*(self.mg/(1-b1**self.t))/(np.sqrt(self.vg/(1-b2**self.t))+eps)
        self.mb=b1*self.mb+(1-b1)*db; self.vb=b2*self.vb+(1-b2)*db**2
        self.beta  -= lr*(self.mb/(1-b1**self.t))/(np.sqrt(self.vb/(1-b2**self.t))+eps)


# ─────────────────────────────────────────────────────────────
#  TDNN LAYER
# ─────────────────────────────────────────────────────────────

class TDNNLayer:
    def __init__(self, in_dim, out_dim, context, lr, use_ln=True):
        self.context = context
        self.ctx_len = context[1]-context[0]+1
        self.lr      = lr
        self.use_ln  = use_ln 
        
        # He Initialization untuk aktivasi ReLU
        fan_in = in_dim * self.ctx_len
        self.W = (np.random.randn(fan_in, out_dim) * np.sqrt(2.0/fan_in)).astype(np.float32)
        self.b = np.zeros(out_dim, dtype=np.float32)
        
        if use_ln:
            self.ln = LayerNorm(out_dim)
            self.ln.lr = lr
            
        self.mW=np.zeros_like(self.W); self.vW=np.zeros_like(self.W)
        self.mb=np.zeros_like(self.b); self.vb=np.zeros_like(self.b)
        self.t=0; self._X=None
        self._dW_acc=np.zeros_like(self.W)
        self._db_acc=np.zeros_like(self.b)
        self._n_acc=0

    def splice(self, X):
        T=X.shape[0]; idx=np.arange(T)
        # Splicing menyusun frame sesuai rentang sub-konteks
        return np.concatenate(
            [X[np.clip(idx+off, 0, T-1)]
             for off in range(self.context[0], self.context[1]+1)], axis=1)

    def forward(self, Xsp):
        self._X = Xsp
        z = Xsp @ self.W + self.b
        if self.use_ln: z = self.ln.forward(z)
        return z

    def backward(self, dout, accumulate=True):
        T = len(dout)
        if self.use_ln: dout = self.ln.backward(dout, accumulate=accumulate)
        dW   = self._X.T @ dout / T
        db   = dout.mean(0)
        dXsp = dout @ self.W.T
        if accumulate:
            self._dW_acc+=dW; self._db_acc+=db; self._n_acc+=1
        else:
            self._adam(dW, db)
            
        B, D_tot = dXsp.shape; D = D_tot // self.ctx_len
        dX = np.zeros((B,D), dtype=np.float32)
        for k in range(self.ctx_len): 
            dX += dXsp[:, k*D : (k+1)*D]
        return dX

    def flush(self):
        if self._n_acc==0: return
        self._adam(self._dW_acc/self._n_acc, self._db_acc/self._n_acc)
        if self.use_ln: self.ln.flush()
        self._dW_acc[:]=0; self._db_acc[:]=0; self._n_acc=0

    def set_lr(self, lr):
        self.lr = lr
        if self.use_ln: self.ln.lr = lr

    def _adam(self, dW, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t+=1
        self.mW=b1*self.mW+(1-b1)*dW; self.vW=b2*self.vW+(1-b2)*dW**2
        self.W-=self.lr*(self.mW/(1-b1**self.t))/(np.sqrt(self.vW/(1-b2**self.t))+eps)
        self.mb=b1*self.mb+(1-b1)*db; self.vb=b2*self.vb+(1-b2)*db**2
        self.b-=self.lr*(self.mb/(1-b1**self.t))/(np.sqrt(self.vb/(1-b2**self.t))+eps)


# ─────────────────────────────────────────────────────────────
#  MODEL TDNN
# ─────────────────────────────────────────────────────────────

class TDNNModel:
    def __init__(self, input_dim, hidden_dims, contexts, output_dim, dropout=0.2, lr=1e-3):
        assert len(hidden_dims) == len(contexts), "Jumlah hidden layer dan context harus sama!"
        self.dropout = dropout; self.training = True
        self.layers = []
        
        in_d = input_dim
        # Buat Input Layer + 4 Hidden Layer sesuai parameter
        for h, ctx in zip(hidden_dims, contexts):
            self.layers.append(TDNNLayer(in_d, h, ctx, lr, use_ln=True))
            in_d = h
            
        # Layer Output Khusus untuk Probability State CTC (Context 0,0 dan tanpa LayerNorm)
        self.layers.append(TDNNLayer(in_d, output_dim, [0, 0], lr, use_ln=False))
        self._rc = []; self._dc = []

    def set_training(self, mode):
        self.training = mode

    def set_lr(self, lr):
        for layer in self.layers: layer.set_lr(lr)

    def forward(self, X):
        self._rc = []; self._dc = []; h = X
        
        # Forward pass melalui semua hidden layers
        for layer in self.layers[:-1]:
            z = layer.forward(layer.splice(h))
            a = np.maximum(0.0, z)  # ReLU
            
            # Terapkan Dropout jika mode Training
            if self.training and self.dropout > 0:
                mask = (np.random.rand(*a.shape) > self.dropout).astype(np.float32)
                # Scaling dilakukan saat forward pass (Inverted Dropout)
                a = a * mask / (1.0 - self.dropout)
            else:
                mask = np.ones(a.shape, dtype=np.float32)
                
            self._rc.append(z); self._dc.append(mask); h = a
            
        # Forward ke Output layer tanpa aktivasi (karena softmax dikalkulasi di CTC Loss)
        out_layer = self.layers[-1]
        return out_layer.forward(out_layer.splice(h))

    def ctc_step(self, X, target, accumulate=True):
        logits = self.forward(X)
        loss, dlogit = ctc_loss_and_grad(logits, target)
        if loss == 0.0: return 0.0
        
        gnorm = np.sqrt((dlogit**2).sum())
        if gnorm > GRAD_CLIP: dlogit = dlogit * (GRAD_CLIP / gnorm)
        
        # Scaling dropout konstanta
        sc = 1.0 / (1.0 - self.dropout + 1e-9) if self.dropout > 0 else 1.0
        dh = self.layers[-1].backward(dlogit, accumulate=accumulate)
        
        for i in reversed(range(len(self.layers)-1)):
            dh = dh * self._dc[i] * sc
            dh = dh * (self._rc[i] > 0).astype(np.float32)  # Derivative of ReLU
            dh = self.layers[i].backward(dh, accumulate=accumulate)
            
        return loss

    def flush(self):
        for layer in self.layers: layer.flush()

    def get_log_probs(self, X):
        self.set_training(False)
        p = _softmax(self.forward(X))
        return np.log(p + 1e-30)

    def save(self, path):
        data = {
            "w": [(l.W.copy(), l.b.copy()) for l in self.layers],
            "ln": [(l.ln.gamma.copy(), l.ln.beta.copy()) if l.use_ln else (None, None) for l in self.layers],
        }
        np.save(path, data, allow_pickle=True)

    def load(self, path):
        d = np.load(path, allow_pickle=True).item()
        for i, (W, b) in enumerate(d["w"]):
            self.layers[i].W = W; self.layers[i].b = b
        if "ln" in d:
            for i, (g, bt) in enumerate(d["ln"]):
                if g is not None and self.layers[i].use_ln:
                    self.layers[i].ln.gamma = g
                    self.layers[i].ln.beta = bt


# ─────────────────────────────────────────────────────────────
#  TRAINING & EVALUATION CODE ...
#  (Sisa kode dari `split_dataset` hingga `main` tetap sama)
# ─────────────────────────────────────────────────────────────

def split_dataset(ids):
    rng=np.random.default_rng(RANDOM_SEED)
    ids=list(ids); perm=rng.permutation(len(ids))
    n=len(ids); ntr=int(n*TRAIN_RATIO); nva=int(n*VAL_RATIO)
    return [ids[i] for i in perm[:ntr]], [ids[i] for i in perm[ntr:ntr+nva]], [ids[i] for i in perm[ntr+nva:]]

def get_lr(epoch, no_improve):
    if epoch<=WARMUP_EPOCH:
        alpha=(epoch-1)/max(WARMUP_EPOCH-1,1)
        return LR_WARMUP+alpha*(LR_INIT-LR_WARMUP)
    return max(LR_INIT*(LR_DECAY**(no_improve//LR_PATIENCE)), LR_MIN)

# Sisanya ikuti implementasi asli Anda pada fungsi train, evaluate, dsb.

  TAHAP 1 — Memuat Data & Ekstraksi MFCC
  Output : D:/doc/2026/FINAL_CODE/src/codeStat/modelling/voiceModelling/srcVoiceModelling/GMM_TDNN/out/test_5
  Membaca D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv ...
  Shape CSV : (1749, 4)
  Kolom     : ['folder', 'subfolder', 'sentences', 'Path_Data']
  Vocab fonem : 34  (+1 blank → total 35)
  Fonem       : ['<sp>', 'a', 'b', 'c', 'd', 'dʰ', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'tʰ', 'u', 'v', 'w', 'x', 'y', 'z', 'è', 'é', 'ê', 'ŋ', 'ɲ']
  Mengekstrak MFCC dari 1749 file .wav ...
    [200/1749] diproses, 0 dilewati
    [400/1749] diproses, 0 dilewati
    [600/1749] diproses, 0 dilewati
    [800/1749] diproses, 0 dilewati
    [1000/1749] diproses, 0 dilewati
    [1200/1749] diproses, 0 dilewati
    [1400/1749] diproses, 0 dilewati
    [1600/1749] diproses, 0 dilewati
    [1749/1749] diproses, 0 dilewati

  Audio berhasil : 1749 | di